In [ ]:
%env PYTHONHASHSEED=0
import os
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
import seaborn as sns
import numpy as np
import anndata as ad
import scanpy as sc
import rpy2
import gseapy
from gseapy import Msigdb

In [ ]:
np.random.seed(0)
sc.set_figure_params(dpi = 300, dpi_save = 300, frameon = False)

In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R
library(ggplot2)
library(ggpubr)
library(SingleCellExperiment)
library(dplyr)
library(edgeR)
library(extrafont)
library(ComplexHeatmap)
loadfonts()

In [ ]:
# load astrocytes anndata object
micros = sc.read_h5ad('../output/human/microglia_scanpy_object_intermediate_postclustering.h5ad')

In [ ]:
micros

In [ ]:
#annotate clusters with cell type labels
micros.obs['final_cluster_names'] = (
    micros.obs["choir_clusters_str"]
    .map(lambda x: {
        '3':'MG-Control',
        '2':'MG-A',
        '5':'MG-B',
        '1':'MG-C1',
        '6':'MG-C2',
        '4':'MG-D'
                   }.get(x, x))
    .astype("category")
)

In [ ]:
# re-order cell type labels
micros.obs['final_cluster_names'] = micros.obs['final_cluster_names'].cat.reorder_categories(['MG-Control',
                                                                                              'MG-A',
                                                                                             'MG-B',
                                                                                             'MG-C1',
                                                                                             'MG-C2',
                                                                                             'MG-D'
                                                                                             ])

In [ ]:
# plot UMAP with sample overlaid
with rc_context({"figure.figsize": (3, 3), "grid.alpha":0}):
    sc.pl.umap(micros, color=['final_cluster_names'], size = 30, frameon=False, title='', legend_loc = None,
              palette = {"MG-Control":"lightgray", "MG-A":"indigo", "MG-B":"violet", "MG-C1":"maroon", "MG-C2":"tomato", "MG-D":"pink"})

In [ ]:
# plot UMAP with sample overlaid
with rc_context({"figure.figsize": (3, 3), "grid.alpha":0}):
    sc.pl.umap(micros, color=['final_cluster_names'], size = 30, frameon=False, title='',
              palette = {"MG-Control":"lightgray", "MG-A":"indigo", "MG-B":"violet", "MG-C1":"maroon", "MG-C2":"tomato", "MG-D":"pink"})

In [ ]:
# plot UMAP with sample overlaid
with rc_context({"figure.figsize": (3, 3), "grid.alpha":0}):
    sc.pl.umap(micros, color=['sampleid'], size = 30, frameon=False, title='', legend_loc = None)

In [ ]:
#annotate clusters with cell type labels
micros.obs['sampleid_nounderscore'] = (
    micros.obs["sampleid"]
    .map(lambda x: {
        'Control_A':'Control A',
        'Control_B':'Control B',
        'Control_C':'Control C',
        'CART_A':'CAR T A',
        'CART_B':'CAR T B',
        'CART_C':'CAR T C',
        'CART_D':'CAR T D',
                   }.get(x, x))
    .astype("category")
)

# re-order cell type labels
micros.obs['sampleid_nounderscore'] = micros.obs['sampleid_nounderscore'].cat.reorder_categories(['CAR T A',
                                                                                                  'CAR T B',
                                                                                                  'CAR T C',
                                                                                                  'CAR T D',
                                                                                                  'Control A',
                                                                                                  'Control B',
                                                                                                  'Control C'
                                                                                             ])

In [ ]:
# plot UMAP with sample overlaid
with rc_context({"figure.figsize": (3, 3), "grid.alpha":0}):
    sc.pl.umap(micros, color=['sampleid_nounderscore'], frameon=False, title='',
               palette = {"CAR T A":"#662506", "CAR T B":"#cc4c02", "CAR T C":"#fb9a29", "CAR T D":"#fee391", 
                         "Control A":"#2171b5", "Control B":"#6baed6", "Control C":"#bdd7e7"}
              )

In [ ]:
%%R
# plot cell type proportions using propeller functions
library(speckle)

In [ ]:
# extract metadata for creating SCE object
cluster_labels = micros.obs['final_cluster_names']
sample_labels = micros.obs['sampleid']
group_labels = micros.obs['condition']

In [ ]:
%%R -i cluster_labels -i sample_labels -i group_labels
# create SCE object
sce <- SingleCellExperiment(list(counts=matrix(ncol = length(sample_labels), nrow = 1)),
                     colData=data.frame(clusters=cluster_labels,
                                        sample=sample_labels,
                                        group=group_labels))

In [ ]:
%%R -w 3 -h 3 -r 300 --units in

color_dict <- c('MG-Control'='#d3d3d3ff', "MG-A"='#4b0082ff', "MG-B"='#ee82eeff', "MG-C1"='#800000ff', "MG-C2"='#ff6347ff', "MG-D"='#ffc0cbff')

# Plot cell type proportions
plotCellTypeProps(clusters = colData(sce)$clusters, sample = colData(sce)$sample) + theme(aspect.ratio = 0.25) +
    scale_x_discrete(limits = c('Control_A', 'Control_B', 'Control_C', 'CART_A', 'CART_B', 'CART_C', 'CART_D')) + 
  scale_fill_manual(values=color_dict) + guides(fill = "none") + 
    theme_pubr() + theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust=1)) + labs(y = "Proportion per Donor", x = "")

In [ ]:
# calculate cell proportions per sample
cell_props = pd.crosstab(micros.obs['final_cluster_names'], micros.obs['sampleid']).apply(lambda col: col / sum(col))
cell_props

In [ ]:
%%R -i cell_props
# import cell proportions dataframe into R and format for making plots
cell_props$celltype <- rownames(cell_props)

cell_props_df <- reshape2::melt(cell_props)

colnames(cell_props_df) <- c("celltype", "sample", "proportion")

cell_props_df <- cell_props_df %>% mutate("condition" = ifelse(sample %in% c("CART_A", "CART_B", "CART_C", "CART_D"), "CAR T", "Control"))

cell_props_df$condition <- factor(cell_props_df$condition, levels = c("Control", "CAR T"))


In [ ]:
%%R
# calculate mean of each group
stats_df = cell_props_df %>% filter(celltype == 'MG-Control') %>% mutate("percent" = proportion*100) %>% group_by(condition) %>% summarise(percent = mean(percent))

stats_df

In [ ]:
%%R -w 6 -h 2 -r 300 --units in
# 
# plot MG-Control cell proportions between conditions
set.seed(1113)
p1 <- ggplot(cell_props_df %>% filter(celltype == 'MG-Control') %>% mutate("percent" = proportion*100), 
        aes(x=condition, y=percent, fill=condition)) +
          geom_jitter(size = 3, width = 0.1, pch=21) + 
            geom_crossbar(data=stats_df, aes(ymin = percent, ymax = percent, color = condition),
                  size=0.5, width = 0.5) + 
            labs(y = "Percentage of microglia in MG-Control", x = "") +
            scale_y_continuous(labels = function(x) paste0(x, "%"), limits = c(0, 105)) + 
                scale_fill_manual(values = c("#9aceeb", "#dc143c")) +
                scale_color_manual(values = c("#9aceeb", "#dc143c")) +
                theme_pubr() + theme(axis.text.x = element_text(size = 14, angle = 0), axis.text.y = element_text(size = 12),
                axis.title.y = element_text(size = 12)) + guides(color = "none", fill = "none") + coord_flip()

ggsave("../output/human/human_microglia_mgcontrol_proportions.pdf", p1, width = 2.5, height = 4, units = "in", dpi = 300)

p1 

In [ ]:
# load scanpro sample-level DA test results
scanpro_df = pd.read_pickle("../output/human/human_microglia_scanpro_differentialabundance_test_results.pkl")

In [ ]:
%%R -i scanpro_df
# reshape to matrix
scanpro_mat = reshape2::acast(scanpro_df, Sample ~ clusters, value.var = "adjusted_p_values")

In [ ]:
%%R
# fill NA values
scanpro_mat[is.na(scanpro_mat)] <- 1

In [ ]:
%%R
# create significance matrices for heatmap annotation
up_sig_mat = scanpro_df %>% mutate(adjusted_p_values = ifelse((prop_ratio > 1) & (adjusted_p_values < 0.01), adjusted_p_values, 1))
up_sig_mat = reshape2::acast(up_sig_mat, Sample~clusters, value.var = "adjusted_p_values")
up_sig_mat = t(up_sig_mat)
up_sig_mat[is.na(up_sig_mat)] <- 1
down_sig_mat = scanpro_df %>% mutate(adjusted_p_values = ifelse((prop_ratio < 1) & (adjusted_p_values < 0.01), adjusted_p_values, 1))
down_sig_mat = reshape2::acast(down_sig_mat, Sample~clusters, value.var = "adjusted_p_values")
down_sig_mat = t(down_sig_mat)
down_sig_mat[is.na(down_sig_mat)] <- 1

In [ ]:
%%R -w 2.5 -h 4 -r 300 --units in
# plot heatmap
hmap = Heatmap(name = '-log10(p-value)', -log10(t(scanpro_mat)), col = circlize::colorRamp2(breaks = c(0, 1, 2, 3), 
                                colors=RColorBrewer::brewer.pal(4, "Purples")),
                            cluster_rows = TRUE, cluster_columns = FALSE, 
                        cell_fun = function(j, i, x, y, w, h, fill){
                                if(up_sig_mat[i, j] < 0.01){
                                	gb = textGrob("↑")
                                	gb_w = convertWidth(grobWidth(gb), "mm")
                                	gb_h = convertHeight(grobHeight(gb), "mm")
                                	grid.text("↑", x, y - gb_h*0.15 + gb_w*0.4, gp = gpar(fontsize = 18, fontface = "bold"))
                                } else if(down_sig_mat[i, j] < 0.01){
                                	gb = textGrob("↓")
                                	gb_w = convertWidth(grobWidth(gb), "mm")
                                	gb_h = convertHeight(grobHeight(gb), "mm")
                                	grid.text("↓", x, y - gb_h*0.15 + gb_w*0.4, gp = gpar(fontsize = 18, fontface = "bold"))
                                } else {
                                    gb = textGrob("")
                                	gb_w = convertWidth(grobWidth(gb), "mm")
                                	gb_h = convertHeight(grobHeight(gb), "mm")
                                    grid.text("", x, y - gb_h*0.5 + gb_w*0.4)
                                }
                            }, 
                        heatmap_legend_param = list(direction = "horizontal", position = "left",
                                                           title_position = "topcenter"
                                                           )
                        )

draw(hmap, heatmap_legend_side = "top")

In [ ]:
micros.write('../output/human/human_microglia_final_annotated_scanpy_object.h5ad')

In [ ]:
# now, run pseudobulk DE testing comparing our CAR T clusters to cluster 3

In [ ]:
micros.X = micros.layers['raw_counts']

In [ ]:
# examine the cell counts in each cluster from each sample
pd.crosstab(micros.obs['choir_clusters_str'], micros.obs['sampleid'])

In [ ]:
# examine the proportion of cells in each cluster from each sample
pd.crosstab(micros.obs['choir_clusters_str'], micros.obs['sampleid']).apply(lambda col: col / sum(col))

In [ ]:
# define function for creating sample pseudobulk aggregates
# adapted from https://www.sc-best-practices.org/conditions/differential_gene_expression.html#pseudobulk
# in this implementation, clusters of large enough size will be automatically split into two or three clusters
# if the number of samples in a the cluster are not ≥3. Note that this is not strictly 'pseudo-bulk'
# DE testing, and will include 'pseudoreplication' in the event that a cluster must be split. 
# We drew this idea from work showing even single sample scRNA-seq datasets benefit from creating pseudo-bulks
# for DE testing (https://doi.org/10.1101/2023.03.28.534443)

import random
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

def aggregate_and_filter(
    adata,
    cell_identity,
    donor_key="sampleid",
    cell_identity_key="cell_type",
    obs_to_keep=['sampleid', 'condition', 'batch', 'sex', 'age'], 
    replicates_per_patient=1,
):
    
    print("Pseudobulk aggregating group " + cell_identity + " ...")

    random.seed(10)
    
    # subset adata to the given cell identity
    adata_cell_pop = adata[adata.obs[cell_identity_key] == cell_identity].copy()
    # check which donors to keep according to the number of cells specified with NUM_OF_CELL_PER_DONOR
    size_by_donor = adata_cell_pop.obs.groupby(by=donor_key).size()
    donors_to_drop = []
    donors_to_drop = [
        donor
        for donor in size_by_donor.index
        if size_by_donor[donor] <= NUM_OF_CELL_PER_DONOR
    ]

    donors_to_keep = [x for x in size_by_donor.index if x not in donors_to_drop]

    n_donors = len([i for i in donors_to_keep])

    prop_df = pd.crosstab(adata.obs[cell_identity_key], adata.obs[donor_key]).apply(lambda col: col / sum(col))
    sig_samples = prop_df.columns[(prop_df[prop_df.index == cell_identity] > 0.1).all()].values

    if n_donors > 1:
        donors_to_keep2 = [x for x in donors_to_keep if x in sig_samples]
    else:
        donors_to_keep2 = donors_to_keep
    
    print("Keeping donors:")
    print(donors_to_keep2)
    
    n_samples = len([i for i in donors_to_keep2])

    split2x = []
    split1x = []

    if n_samples >= 3:
        print("Cluster has 3+ samples. Proceeding.")
    
    if n_samples == 2:
        print("Less than 3 samples in cluster. Attempting to subdivide largest sample into 2 sub-samples...")
        
        largest_sample = size_by_donor[[i for i in size_by_donor.index]].idxmax()
        
        if size_by_donor[largest_sample] >= (2*NUM_OF_CELL_PER_DONOR):
            print("Splitting sample: ")
            print(largest_sample)
            split1x.append(largest_sample)

        else:
            print("No sample in this cluster is large enough to split. Proceeding with 2 samples only.")
            print("WARNING: DE TESTING WILL BE UNDER-POWERED.")
            
    if n_samples == 1:
        print("Only 1 sample is in this cluster. Attempting to subdivide this sample into 3 sub-samples...")
    
        solo_sample = [i for i in donors_to_keep][0]
        
        if size_by_donor[solo_sample] >= (3*NUM_OF_CELL_PER_DONOR):
            print("Splitting sample into 3 sub-samples: ")
            print(solo_sample)
            split2x.append(solo_sample)
        elif size_by_donor[solo_sample] >= (2*NUM_OF_CELL_PER_DONOR):
            print("Sample is not large enough to split into 3 sub-samples. Splitting sample into 2 sub-samples:")
            print(solo_sample)
            print("WARNING: DE TESTING WILL BE UNDER-POWERED.")
            split1x.append(solo_sample)
        else:
            print("Sample is too small to by sub-divided:")
            print(solo_sample)
            print("WARNING: CANNOT USE THIS SINGLE PSEUDOBULK FOR DE TESTING.")

    if n_samples == 0:
        print("This cluster has NO samples of appropriate size...")
    
    df = pd.DataFrame(columns=[*adata_cell_pop.var_names, *obs_to_keep])

    adata_cell_pop = adata_cell_pop[adata_cell_pop.obs[donor_key].isin(donors_to_keep2)]

    adata_cell_pop.obs[donor_key] = adata_cell_pop.obs[donor_key].astype("category")
    for i, donor in enumerate(donors := adata_cell_pop.obs[donor_key].cat.categories):
        print(f"\tProcessing donor {i+1} out of {len(donors)}...", end="\r")
        if donor in donors_to_keep2:
            adata_donor = adata_cell_pop[adata_cell_pop.obs[donor_key] == donor]
            # create replicates for each donor
            indices = list(adata_donor.obs_names)
            random.shuffle(indices)
            if donor in split2x:
                replicates_per_patient = 3
            elif donor in split1x:
                replicates_per_patient = 2
            else:
                replicates_per_patient = 1
                
            indices = np.array_split(np.array(indices), replicates_per_patient)
            for i, rep_idx in enumerate(indices):
                adata_replicate = adata_donor[rep_idx]
                # specify how to aggregate: sum gene expression for each gene for each donor and also keep the condition information
                agg_dict = {gene: "sum" for gene in adata_replicate.var_names}
                for obs in obs_to_keep:
                    agg_dict[obs] = "first"
                # create a df with all genes, donor and condition info
                df_donor = pd.DataFrame(adata_replicate.layers['raw_counts'].A)
                df_donor.index = adata_replicate.obs_names
                df_donor.columns = adata_replicate.var_names
                df_donor = df_donor.join(adata_replicate.obs[obs_to_keep])
                # aggregate
                df_donor = df_donor.groupby(by=donor_key).agg(agg_dict)
                df_donor[donor_key] = donor
                df.loc[f"donor_{donor}_{i}"] = df_donor.loc[donor]
    print("\n")
    # create AnnData object from the df
    adata_cell_pop = sc.AnnData(
        df[adata_cell_pop.var_names], obs=df.drop(columns=adata_cell_pop.var_names)   
    )

    adata_cell_pop.obs['cluster'] = cell_identity 
    
    return adata_cell_pop

In [ ]:
# run pseudobulk aggregation
NUM_OF_CELL_PER_DONOR = 15

micros_2_pbs = aggregate_and_filter(micros, cell_identity = '2', cell_identity_key = "choir_clusters_str")
micros_5_pbs = aggregate_and_filter(micros, cell_identity = '5', cell_identity_key = "choir_clusters_str")
micros_1_pbs = aggregate_and_filter(micros, cell_identity = '1', cell_identity_key = "choir_clusters_str")
micros_6_pbs = aggregate_and_filter(micros, cell_identity = '6', cell_identity_key = "choir_clusters_str")
micros_4_pbs = aggregate_and_filter(micros, cell_identity = '4', cell_identity_key = "choir_clusters_str")
micros_3_pbs = aggregate_and_filter(micros, cell_identity = '3', cell_identity_key = "choir_clusters_str")

In [ ]:
## concatenate all pseudobulk anndatas
micros_pbs = ad.concat([micros_2_pbs,
                       micros_5_pbs,
                       micros_1_pbs,
                        micros_6_pbs,
                       micros_4_pbs,
                       micros_3_pbs])
micros_pbs.obs_names_make_unique()

In [ ]:
micros_pbs

In [ ]:
micros_pbs.obs

In [ ]:
# extract count matrix and metadata info for passing to R
input_matrix = micros_pbs.X
gene_ids = micros_pbs.var_names.values
condition_ids = micros_pbs.obs['condition']
cluster_ids = micros_pbs.obs['cluster']
sample_ids = micros_pbs.obs['sampleid'].values
batch_ids = micros_pbs.obs['batch'].values
sex_ids = micros_pbs.obs['sex'].values

In [ ]:
%%R -i input_matrix -i gene_ids -i sample_ids -i batch_ids -i cluster_ids -i condition_ids -i sex_ids
# import to R and check dimensions
colnames(input_matrix) <- paste(sample_ids, cluster_ids, sep = "_")
rownames(input_matrix) <- gene_ids
# create SCE object
micros_pbs <- SingleCellExperiment(list('counts'=input_matrix))
# add metadata to SCE object
colData(micros_pbs)$sample <- sample_ids
colData(micros_pbs)$cluster <- cluster_ids
colData(micros_pbs)$batch <- batch_ids
colData(micros_pbs)$condition <- condition_ids
colData(micros_pbs)$sex <- sex_ids

In [ ]:
%%R
# set factor levels for cluster metadata
colData(micros_pbs)$cluster <- factor(colData(micros_pbs)$cluster, levels = c("3", "2", "5", "1", "6", "4"))

In [ ]:
%%R
# define function for edgeR differential expression testing
fit_model <- function(adata_){

    set.seed(1)
    
    # create an edgeR object with counts and grouping factor
    group <- colData(adata_)$cluster
    batch <- colData(adata_)$batch
    sex <- colData(adata_)$sex

    y <- DGEList(assay(adata_, "counts"), group = group)

    # filter out genes with low counts
    print("Dimensions before subsetting:")
    print(dim(y))
    print("")
    keep <- filterByExpr(y)
    y <- y[keep, , keep.lib.sizes=FALSE]
    print("Dimensions after subsetting:")
    print(dim(y))
    print("")

    # normalize
    y <- calcNormFactors(y)
    # create a design matrix:
    design <- model.matrix(~ group + batch + sex)
    # estimate dispersion
    y <- estimateDisp(y, design = design)
    # fit the model
    fit <- glmQLFit(y, design)
    return(list("fit"=fit, "design"=design, "y"=y))
}

In [ ]:
%%time
%%R
# run edgeR
outs <-fit_model(micros_pbs)

In [ ]:
%%R
# extract results
fit <- outs$fit
y <- outs$y

In [ ]:
%%R -w 12 -h 5 --units in -r 300
# create MDS plot
plotMDS(y, col=ifelse(y$samples$group == "3", "blue", "red"))

In [ ]:
%%R

plotQLDisp(fit)

In [ ]:
%%R

# save output
saveRDS(micros_pbs, "../output/human/human_microglia_edgeR_pseudobulk_sce_object.rds")

In [ ]:
%%R
# save output
saveRDS(outs, "../output/human/human_microglia_edgeR_glmqlf_output_object.rds")

In [ ]:
%%R -o edgeR_filtered_genes

edgeR_filtered_genes = rownames(outs$y$counts)

In [ ]:
np.save('../output/human/human_microglia_edgeR_filtered_genes.npy', edgeR_filtered_genes)

In [ ]:
# Test Cluster 2 versus Cluster 3

In [ ]:
%%R
# run glmQLFTest
qlf_group2 <- glmQLFTest(fit, coef="group2")

In [ ]:
%%R -o tt_group2
# get all of the DE genes and calculate Benjamini-Hochberg adjusted FDR
tt_group2 <- topTags(qlf_group2, n = Inf)
tt_group2 <- tt_group2$table

In [ ]:
# view DE results
tt_group2[(tt_group2.FDR < 0.05/5) & (abs(tt_group2.logFC) > 1)]

In [ ]:
%%R

saveRDS(qlf_group2, "../output/human/human_microglia_edgeR_glmqlftest_output_object_cluster2vcluster3.rds")

In [ ]:
# save DE results to CSV and pickle files
tt_group2.to_csv("../output/human/human_microglia_cluster2_v_cluster3_edgeR_results.csv")
tt_group2.to_pickle("../output/human/human_microglia_cluster2_v_cluster3_edgeR_results.pkl")
tt_group2[(tt_group2.FDR < 0.05/5) & (abs(tt_group2.logFC) > 1)].to_csv("../output/human/human_microglia_cluster2_v_cluster3_edgeR_significant_results.csv")
tt_group2[(tt_group2.FDR < 0.05/5) & (abs(tt_group2.logFC) > 1)].to_pickle("../output/human/human_microglia_cluster2_v_cluster3_edgeR_significant_results.pkl")

In [ ]:
%%R
# create log fold change versus average expression plot
plotSmear(qlf_group2, de.tags = rownames(tt_group2)[which((tt_group2$FDR<0.05/5) & (abs(tt_group2$logFC) > 1))])

In [ ]:
%%R -w 4 -h 3 --units in -r 300

select_genes <- c('SPP1', 'TMEM163', 'P2RY12', 'CX3CR1', 'NAV2', 
                 'TNF', 'IL1B', 'TNFAIP3', 'MSR1', 'ACTG1', 'FOS', 'JUN',
                 'S100A9', 'BCL2A1', 'VEGFA', 'SLC2A3', 'CD163', 'MS4A7', 'CD14',
                 'SCARB1', 'SELPLG', 'STK10')


# plot volcano plot with automatically generated gene labels
p1 <- ggplot(data = tt_group2 %>% mutate(gene = rownames(.)) %>% mutate(label = ifelse((abs(logFC) > 1) & (FDR < (0.05/5)) & (gene %in% select_genes), gene, NA)) %>% 
                                                       mutate(significance = ifelse((abs(logFC) > 1) & (FDR < 0.05/5), "Significant", "NS")), 
              aes(x = logFC, y = -log10(FDR), label = label, color = significance)) +
  geom_vline(xintercept = c(-1, 1), col = "gray", linetype = 'dashed') +
  geom_hline(yintercept = -log10(0.05/5), col = "gray", linetype = 'dashed') +
  geom_point(size = 0.25) + scale_colour_manual(name = 'Significance', values = c("black", "red2")) + 
    ggrepel::geom_text_repel(max.overlaps = 30, min.segment.length = unit(0, 'lines'), nudge_y = .5, nudge_x = 0.5, aes(fontface="italic", color = NULL), force = 3, seed = 72) + 
    labs(x = "Log2-fold change") + 
    theme_pubr() + theme(axis.title.x = element_text(size = 14), axis.title.y = element_text(size = 14), text = element_text(family = "Arial")) + 
    guides(color = "none")

p1

In [ ]:
# Test Cluster 5 vs Cluster 3

In [ ]:
%%R
# run glmQLFTest
qlf_group5 <- glmQLFTest(fit, coef="group5")

In [ ]:
%%R -o tt_group5
# get all of the DE genes and calculate Benjamini-Hochberg adjusted FDR
tt_group5 <- topTags(qlf_group5, n = Inf)
tt_group5 <- tt_group5$table

In [ ]:
# view DE results
tt_group5[(tt_group5.FDR < 0.05/5) & (abs(tt_group5.logFC) > 1)]

In [ ]:
%%R

saveRDS(qlf_group5, "../output/human/human_microglia_edgeR_glmqlftest_output_object_cluster5vcluster3.rds")

In [ ]:
# save DE results to CSV and pickle files
tt_group5.to_csv("../output/human/human_microglia_cluster5_v_cluster3_edgeR_results.csv")
tt_group5.to_pickle("../output/human/human_microglia_cluster5_v_cluster3_edgeR_results.pkl")
tt_group5[(tt_group5.FDR < 0.05/5) & (abs(tt_group5.logFC) > 1)].to_csv("../output/human/human_microglia_cluster5_v_cluster3_edgeR_significant_results.csv")
tt_group5[(tt_group5.FDR < 0.05/5) & (abs(tt_group5.logFC) > 1)].to_pickle("../output/human/human_microglia_cluster5_v_cluster3_edgeR_significant_results.pkl")

In [ ]:
%%R
# create log fold change versus average expression plot
plotSmear(qlf_group5, de.tags = rownames(tt_group5)[which((tt_group5$FDR<0.05/5) & (abs(tt_group5$logFC) > 1))])

In [ ]:
%%R -w 4 -h 3 --units in -r 300

select_genes <- c('SPP1', 'TMEM163', 'P2RY12', 'CX3CR1', 'NAV2', 
                 'CD14', 'APOE', 'CD63', 'CTSB', 'FTH1', 'FTL', 
                  'C1QB', 'C1QC', 'RPL35A', 'RPL34', 'PLCXD3',
                  'AKT3', 'OXR1', 'ABCC4'
                 )

# plot volcano plot with automatically generated gene labels
p1 <- ggplot(data = tt_group5 %>% mutate(gene = rownames(.)) %>% mutate(label = ifelse((abs(logFC) > 1) & (FDR < (0.05/5)) & (gene %in% select_genes), gene, NA)) %>% 
                                                       mutate(significance = ifelse((abs(logFC) > 1) & (FDR < 0.05/5), "Significant", "NS")), 
              aes(x = logFC, y = -log10(FDR), label = label, color = significance)) +
  geom_vline(xintercept = c(-1, 1), col = "gray", linetype = 'dashed') +
  geom_hline(yintercept = -log10(0.05/5), col = "gray", linetype = 'dashed') +
  geom_point(size = 0.25) + scale_colour_manual(name = 'Significance', values = c("black", "red2")) + 
    ggrepel::geom_text_repel(max.overlaps = 30, min.segment.length = unit(0, 'lines'), nudge_y = .15, nudge_x = 0.2, aes(fontface="italic", color = NULL), force = 3, seed = 72) + 
    labs(x = "Log2-fold change") + 
    theme_pubr() + theme(axis.title.x = element_text(size = 14), axis.title.y = element_text(size = 14), text = element_text(family = "Arial")) + 
    guides(color = "none")

p1

In [ ]:
# Test Cluster 1 versus Cluster 3

In [ ]:
%%R
# run glmQLFTest
qlf_group1 <- glmQLFTest(fit, coef="group1")

In [ ]:
%%R -o tt_group1
# get all of the DE genes and calculate Benjamini-Hochberg adjusted FDR
tt_group1 <- topTags(qlf_group1, n = Inf)
tt_group1 <- tt_group1$table

In [ ]:
# view DE results
tt_group1[(tt_group1.FDR < 0.05/5) & (abs(tt_group1.logFC) > 1)]

In [ ]:
%%R

saveRDS(qlf_group1, "../output/human/human_microglia_edgeR_glmqlftest_output_object_cluster1vcluster3.rds")

In [ ]:
# save DE resuls to CSV and pickle files
tt_group1.to_csv("../output/human/human_microglia_cluster1_v_cluster3_edgeR_results.csv")
tt_group1.to_pickle("../output/human/human_microglia_cluster1_v_cluster3_edgeR_results.pkl")
tt_group1[(tt_group1.FDR < 0.05/5) & (abs(tt_group1.logFC) > 1)].to_csv("../output/human/human_microglia_cluster1_v_cluster3_edgeR_significant_results.csv")
tt_group1[(tt_group1.FDR < 0.05/5) & (abs(tt_group1.logFC) > 1)].to_pickle("../output/human/human_microglia_cluster1_v_cluster3_edgeR_significant_results.pkl")

In [ ]:
%%R
# create log fold change versus average expression plot
plotSmear(qlf_group1, de.tags = rownames(tt_group1)[which((tt_group1$FDR<0.05/5) & (abs(tt_group1$logFC) > 1))])

In [ ]:
%%R -w 4 -h 3 --units in -r 300

select_genes <- c('SPP1', 'TMEM163', 'P2RY12', 'CX3CR1', 'NAV2', 
                 'CD14', 'CDKN1A', 'IL1B', 'SOCS3', 'CEBPD',
                  'FKBP5', 'PRKCA', 'CTSC', 'WNT5A',
                  'ACSL3', 'P2RY13'
                 )

# plot volcano plot with automatically generated gene labels
p1 <- ggplot(data = tt_group1 %>% mutate(gene = rownames(.)) %>% mutate(label = ifelse((abs(logFC) > 1) & (FDR < (0.05/5)) & (gene %in% select_genes), gene, NA)) %>% 
                                                       mutate(significance = ifelse((abs(logFC) > 1) & (FDR < 0.05/5), "Significant", "NS")), 
              aes(x = logFC, y = -log10(FDR), label = label, color = significance)) +
  geom_vline(xintercept = c(-1, 1), col = "gray", linetype = 'dashed') +
  geom_hline(yintercept = -log10(0.05/5), col = "gray", linetype = 'dashed') +
  geom_point(size = 0.25) + scale_colour_manual(name = 'Significance', values = c("black", "red2")) + 
    ggrepel::geom_text_repel(max.overlaps = 30, min.segment.length = unit(0, 'lines'), nudge_y = .6, nudge_x = 0.1, aes(fontface="italic", color = NULL), force = 3, seed = 72) + 
    labs(x = "Log2-fold change") + 
    theme_pubr() + theme(axis.title.x = element_text(size = 14), axis.title.y = element_text(size = 14), text = element_text(family = "Arial")) + 
    guides(color = "none")

p1

In [ ]:
# Test Cluster 6 versus Cluster 3

In [ ]:
%%R
# run glmQLFTest
qlf_group6 <- glmQLFTest(fit, coef="group6")

In [ ]:
%%R -o tt_group6
# get all of the DE genes and calculate Benjamini-Hochberg adjusted FDR
tt_group6 <- topTags(qlf_group6, n = Inf)
tt_group6 <- tt_group6$table

In [ ]:
# view DE results
tt_group6[(tt_group6.FDR < 0.05/5) & (abs(tt_group6.logFC) > 1)]

In [ ]:
%%R

saveRDS(qlf_group6, "../output/human/human_microglia_edgeR_glmqlftest_output_object_cluster6vcluster3.rds")

In [ ]:
# save DE resuls to CSV and pickle files
tt_group6.to_csv("../output/human/human_microglia_cluster6_v_cluster3_edgeR_results.csv")
tt_group6.to_pickle("../output/human/human_microglia_cluster6_v_cluster3_edgeR_results.pkl")
tt_group6[(tt_group6.FDR < 0.05/5) & (abs(tt_group6.logFC) > 1)].to_csv("../output/human/human_microglia_cluster6_v_cluster3_edgeR_significant_results.csv")
tt_group6[(tt_group6.FDR < 0.05/5) & (abs(tt_group6.logFC) > 1)].to_pickle("../output/human/human_microglia_cluster6_v_cluster3_edgeR_significant_results.pkl")

In [ ]:
%%R
# create log fold change versus average expression plot
plotSmear(qlf_group6, de.tags = rownames(tt_group6)[which((tt_group6$FDR<0.05/5) & (abs(tt_group6$logFC) > 1))])

In [ ]:
%%R -w 4 -h 3 --units in -r 300


select_genes <- c('SPP1', 'TMEM163', 'P2RY12', 'CX3CR1', 'NAV2', 
                 'CD14', 'CDKN1A', 'IL1B', 'SOCS3', 'CEBPD',
                  'BCL6', 'SLC2A3', 'SLC2A5', 'ARID5A', 'ADAM28',
                  'PARP1', 'SNX2'
                 )

# plot volcano plot with automatically generated gene labels
p1 <- ggplot(data = tt_group6 %>% mutate(gene = rownames(.)) %>% mutate(label = ifelse((abs(logFC) > 1) & (FDR < (0.05/5)) & (gene %in% select_genes), gene, NA)) %>% 
                                                       mutate(significance = ifelse((abs(logFC) > 1) & (FDR < 0.05/5), "Significant", "NS")), 
              aes(x = logFC, y = -log10(FDR), label = label, color = significance)) +
  geom_vline(xintercept = c(-1, 1), col = "gray", linetype = 'dashed') +
  geom_hline(yintercept = -log10(0.05/5), col = "gray", linetype = 'dashed') +
  geom_point(size = 0.25) + scale_colour_manual(name = 'Significance', values = c("black", "red2")) + 
    ggrepel::geom_text_repel(max.overlaps = 30, min.segment.length = unit(0, 'lines'), nudge_y = .2, nudge_x = -0.3, aes(fontface="italic", color = NULL), force = 3, seed = 72) + 
    labs(x = "Log2-fold change") + 
    theme_pubr() + theme(axis.title.x = element_text(size = 14), axis.title.y = element_text(size = 14), text = element_text(family = "Arial")) + 
    guides(color = "none")

p1

In [ ]:
# Test Cluster 4 versus Cluster 3

In [ ]:
%%R
# run glmQLFTest
qlf_group4 <- glmQLFTest(fit, coef="group4")

In [ ]:
%%R -o tt_group4
# get all of the DE genes and calculate Benjamini-Hochberg adjusted FDR
tt_group4 <- topTags(qlf_group4, n = Inf)
tt_group4 <- tt_group4$table

In [ ]:
# view DE results
tt_group4[(tt_group4.FDR < 0.05/5) & (abs(tt_group4.logFC) > 1)]

In [ ]:
%%R

saveRDS(qlf_group4, "../output/human/human_microglia_edgeR_glmqlftest_output_object_cluster4vcluster3.rds")

In [ ]:
# save DE resuls to CSV and pickle files
tt_group4.to_csv("../output/human/human_microglia_cluster4_v_cluster3_edgeR_results.csv")
tt_group4.to_pickle("../output/human/human_microglia_cluster4_v_cluster3_edgeR_results.pkl")
tt_group4[(tt_group4.FDR < 0.05/5) & (abs(tt_group4.logFC) > 1)].to_csv("../output/human/human_microglia_cluster4_v_cluster3_edgeR_significant_results.csv")
tt_group4[(tt_group4.FDR < 0.05/5) & (abs(tt_group4.logFC) > 1)].to_pickle("../output/human/human_microglia_cluster4_v_cluster3_edgeR_significant_results.pkl")

In [ ]:
%%R
# create log fold change versus average expression plot
plotSmear(qlf_group4, de.tags = rownames(tt_group4)[which((tt_group4$FDR<0.05/5) & (abs(tt_group4$logFC) > 1))])

In [ ]:
%%R -w 4 -h 3 --units in -r 300

select_genes <- c('SPP1', 'TMEM163', 'P2RY12', 'CX3CR1', 'NAV2', 
                  'BACH2', 'CXCR4', 'GBP2', 'HILPDA', 'WARS1',
                  'PDK3', 'TPI1', 'STAT1', 'DOCK10', 'ABCC5',
                  'TLE4', 'TRAF3IP3'
                 )

# plot volcano plot with automatically generated gene labels
p1 <- ggplot(data = tt_group4 %>% mutate(gene = rownames(.)) %>% mutate(label = ifelse((abs(logFC) > 1) & (FDR < (0.05/5)) & (gene %in% select_genes), gene, NA)) %>% 
                                                       mutate(significance = ifelse((abs(logFC) > 1) & (FDR < 0.05/5), "Significant", "NS")), 
              aes(x = logFC, y = -log10(FDR), label = label, color = significance)) +
  geom_vline(xintercept = c(-1, 1), col = "gray", linetype = 'dashed') +
  geom_hline(yintercept = -log10(0.05/5), col = "gray", linetype = 'dashed') +
  geom_point(size = 0.25) + scale_colour_manual(name = 'Significance', values = c("black", "red2")) + 
    ggrepel::geom_text_repel(max.overlaps = 30, min.segment.length = unit(0, 'lines'), nudge_y = .5, nudge_x = 0.7, aes(fontface="italic", color = NULL), force = 3, seed = 72) + 
    labs(x = "Log2-fold change") + 
    theme_pubr() + theme(axis.title.x = element_text(size = 14), axis.title.y = element_text(size = 14), text = element_text(family = "Arial")) + 
    guides(color = "none")

p1

In [ ]:
%%R


edgeR_df <- do.call("rbind", list(
    as.data.frame(tt_group2) %>% mutate(gene = rownames(.)) %>% mutate(cluster = "MG-A"),
    as.data.frame(tt_group5) %>% mutate(gene = rownames(.)) %>% mutate(cluster = "MG-B"),
    as.data.frame(tt_group1) %>% mutate(gene = rownames(.)) %>% mutate(cluster = "MG-C1"),
    as.data.frame(tt_group6) %>% mutate(gene = rownames(.)) %>% mutate(cluster = "MG-C2"),
    as.data.frame(tt_group4) %>% mutate(gene = rownames(.)) %>% mutate(cluster = "MG-D")
    ))


sig_degs_df <- do.call("rbind", list(
    as.data.frame(tt_group2) %>% filter(abs(logFC) > 1) %>% mutate(gene = rownames(.)) %>% mutate(cluster = "MG-A") %>% filter(FDR < 0.05/5),
    as.data.frame(tt_group5) %>% filter(abs(logFC) > 1) %>% mutate(gene = rownames(.)) %>% mutate(cluster = "MG-B") %>% filter(FDR < 0.05/5),
    as.data.frame(tt_group1) %>% filter(abs(logFC) > 1) %>% mutate(gene = rownames(.)) %>% mutate(cluster = "MG-C1") %>% filter(FDR < 0.05/5),
    as.data.frame(tt_group6) %>% filter(abs(logFC) > 1) %>% mutate(gene = rownames(.)) %>% mutate(cluster = "MG-C2") %>% filter(FDR < 0.05/5),
    as.data.frame(tt_group4) %>% filter(abs(logFC) > 1) %>% mutate(gene = rownames(.)) %>% mutate(cluster = "MG-D") %>% filter(FDR < 0.05/5)
    ))

up_degs <- list(
    "Cluster2"=as.data.frame(tt_group2) %>% filter(logFC > 1) %>% filter(FDR < 0.05/5) %>% rownames(.),
    "Cluster5"=as.data.frame(tt_group5) %>% filter(logFC > 1) %>% filter(FDR < 0.05/5) %>% rownames(.),
    "Cluster1"=as.data.frame(tt_group1) %>% filter(logFC > 1) %>% filter(FDR < 0.05/5) %>% rownames(.),
    "Cluster6"=as.data.frame(tt_group6) %>% filter(logFC > 1) %>% filter(FDR < 0.05/5) %>% rownames(.),
    "Cluster4"=as.data.frame(tt_group4) %>% filter(logFC > 1) %>% filter(FDR < 0.05/5) %>% rownames(.)
    )

down_degs <- list(
    "Cluster2"=as.data.frame(tt_group2) %>% filter(logFC < -1) %>% filter(FDR < 0.05/5) %>% rownames(.),
    "Cluster5"=as.data.frame(tt_group5) %>% filter(logFC < -1) %>% filter(FDR < 0.05/5) %>% rownames(.),
    "Cluster1"=as.data.frame(tt_group1) %>% filter(logFC < -1) %>% filter(FDR < 0.05/5) %>% rownames(.),
    "Cluster6"=as.data.frame(tt_group6) %>% filter(logFC < -1) %>% filter(FDR < 0.05/5) %>% rownames(.),
    "Cluster4"=as.data.frame(tt_group4) %>% filter(logFC < -1) %>% filter(FDR < 0.05/5) %>% rownames(.)
    )


In [ ]:
%%R -o ann_sig_df
# examine the number of DEGs in each cluster compared to MG-control
ann_sig_df = sig_degs_df %>% mutate(annotation = ifelse(logFC > 1, "Up-regulated", "Down-regulated"))

In [ ]:
data = pd.crosstab(ann_sig_df.cluster, ann_sig_df.annotation)

In [ ]:
data

In [ ]:
font_color = '#525252'
hfont = {'fontname':'Arial'}
facecolor = '#eaeaf2'
color1 = '#29ABE2'
color2 = '#ED1C24'
index = data.index
column0 = data['Down-regulated']
column1 = data['Up-regulated']
title0 = 'Downregulated'
title1 = 'Upregulated'

fig, axes = plt.subplots(figsize=(10,8), ncols=2, sharey=True)
fig.tight_layout()

axes[0].barh(index, column0, align='center', color=color1, zorder=10)
axes[0].set_title(title0, fontsize=25, pad=15, color=color1, fontweight="bold", **hfont)
axes[1].barh(index, column1, align='center', color=color2, zorder=10)
axes[1].set_title(title1, fontsize=25, pad=15, color=color2, fontweight="bold", **hfont)

# If you have positive numbers and want to invert the x-axis of the left plot
axes[0].invert_xaxis() 

# To show data from highest to lowest
plt.gca().invert_yaxis()

axes[0].set(yticks=data.index, yticklabels=data.index)
axes[0].yaxis.tick_left()
axes[0].tick_params(axis='y', colors='white') # tick color

# Hide tick marks
axes[1].tick_params(axis='y', bottom=False, top=False, left=False, right=False)

# Show only tick labels
plt.xticks(visible=True)
plt.yticks(visible=True)



for label in (axes[0].get_xticklabels()):
    label.set(fontsize=15, color='black', **hfont)
for label in (axes[0].get_yticklabels()):
    label.set(fontsize=22, color='black', fontweight="bold", **hfont)
for label in (axes[1].get_xticklabels()):
    label.set(fontsize=15, color='black', **hfont)
for label in (axes[1].get_yticklabels()):
    label.set(fontsize=22, color='black', fontweight="bold", **hfont)

axes[0].grid(False)
axes[1].grid(False)

plt.subplots_adjust(wspace=0, top=0.85, bottom=0.1, left=0.18, right=0.95)


In [ ]:
%%R

up_mat = make_comb_mat(up_degs)
down_mat = make_comb_mat(down_degs)

In [ ]:
%%R

head(up_mat)

In [ ]:
%%R

head(down_mat)

In [ ]:
%%R

UpSet(up_mat)

In [ ]:
%%R

UpSet(down_mat)

In [ ]:
%%R

print("Up-regulated DEGs common to all samples:")
print(extract_comb(up_mat, "11111"))
print(noquote(""))
print("Down-regulated DEGs common to all samples:")
print(extract_comb(down_mat, "11111"))

In [ ]:
from matplotlib.ticker import MaxNLocator

In [ ]:
with plt.rc_context({"axes.grid": False, "figure.figsize":[6, 1.5]}):
    ax = sc.pl.violin(micros, keys = ['SPP1'], layer = "log1p_norm", 
                 groupby = "choir_clusters_str",
                 order = ['3', '2', '5', '1', '6', '4'],
                 palette = {"3":"lightgray", "2":"indigo", "5":"violet", "1":"maroon", "6":"tomato", "4":"pink"},
                 xlabel = None,
                 size = 0.5,use_raw = False, show = False)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True, nbins = 2))
    plt.ylabel(ylabel = 'SPP1', fontdict={'fontstyle': 'italic'})

In [ ]:
with plt.rc_context({"axes.grid": False, "figure.figsize":[6, 1.5]}):
    ax = sc.pl.violin(micros, keys = ['TMEM163'], layer = "log1p_norm", 
                 groupby = "choir_clusters_str",
                 order = ['3', '2', '5', '1', '6', '4'],
                 palette = {"3":"lightgray", "2":"indigo", "5":"violet", "1":"maroon", "6":"tomato", "4":"pink"},
                 xlabel = None,
                 size = 0.5,use_raw = False, show = False)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True, nbins = 2))
    plt.ylabel(ylabel = 'TMEM163', fontdict={'fontstyle': 'italic'})

In [ ]:
with plt.rc_context({"axes.grid": False, "figure.figsize":[6, 1.5]}):
    ax = sc.pl.violin(micros, keys = ['P2RY12'], layer = "log1p_norm", 
                 groupby = "choir_clusters_str",
                 order = ['3', '2', '5', '1', '6', '4'],
                 palette = {"3":"lightgray", "2":"indigo", "5":"violet", "1":"maroon", "6":"tomato", "4":"pink"},
                 xlabel = None,
                 size = 0.5,use_raw = False, show = False)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True, nbins = 2))
    plt.ylabel(ylabel = 'P2RY12', fontdict={'fontstyle': 'italic'})

In [ ]:
with plt.rc_context({"axes.grid": False, "figure.figsize":[6, 1.5]}):
    ax = sc.pl.violin(micros, keys = ['CX3CR1'], layer = "log1p_norm", 
                 groupby = "choir_clusters_str",
                 order = ['3', '2', '5', '1', '6', '4'],
                 palette = {"3":"lightgray", "2":"indigo", "5":"violet", "1":"maroon", "6":"tomato", "4":"pink"},
                 xlabel = None,
                 size = 0.5,use_raw = False, show = False)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True, nbins = 2))
    plt.ylabel(ylabel = 'CX3CR1', fontdict={'fontstyle': 'italic'})

In [ ]:
with plt.rc_context({"axes.grid": False, "figure.figsize":[6, 1.5]}):
    ax = sc.pl.violin(micros, keys = ['NAV2'], layer = "log1p_norm", 
                 groupby = "choir_clusters_str",
                 order = ['3', '2', '5', '1', '6', '4'],
                 palette = {"3":"lightgray", "2":"indigo", "5":"violet", "1":"maroon", "6":"tomato", "4":"pink"},
                 xlabel = None,
                 size = 0.5,use_raw = False, show = False)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True, nbins = 2))
    plt.ylabel(ylabel = 'NAV2', fontdict={'fontstyle': 'italic'})

In [ ]:
%%R
# extract significant DEGs for creating a heatmap of LFCs
sig_degs = edgeR_df %>% filter(logFC > 1) %>% filter(FDR < 0.05/5) %>% .$gene

In [ ]:
%%R

filt_edgeR_df = edgeR_df %>% filter(gene %in% sig_degs)

In [ ]:
%%R
# because we want to show only significant fold changes, we impute any fold change which doesn't meet our significance threshold with 0
filt_edgeR_df = filt_edgeR_df %>% mutate(logFC = ifelse(FDR > 0.05/5, 0, logFC))

In [ ]:
%%R
# remove unnecessary columns
filt_edgeR_df = filt_edgeR_df %>% select(c('cluster', 'gene', 'logFC'))

In [ ]:
%%R
# reshape into matrix
logFC_df = reshape2::acast(filt_edgeR_df, gene ~ cluster, value.var = "logFC")

In [ ]:
%%R

library(seriation)

In [ ]:
%%R
set.seed(123)
seri_order<- c(seriate(dist(logFC_df, method = "minkowski"), method = "TSP"),
  seriate(dist(t(logFC_df), method = "minkowski"), method = "TSP"))

In [ ]:
%%R
# reorder matrix for heatmap plotting
logFC_df_ser = logFC_df[get_order(seri_order, 1), get_order(seri_order, 2)]

In [ ]:
%%R
# create gene annotation highlighting select genes
selectGenes <- c("SPP1", "TMEM163", "TNF", "IL1B", "C1QA", "C1QB", "C1QC", 
                "TNFAIP3", "MSR1", "ACTG1", "SOCS3", "CEBPB", "CEBPD", 
                "GBP2", "HILPDA", "WARS1", "PDK3", "TPI1", "CXCR4", 
                "APOE", "CD63", "CTSB", "FTH1", "FTL", "RPL35A", "RPL34", 
                "STAT3", "STAT1", "STAT5A", "CD14", "FOS", "JUNB", "SLC2A3", 
                "LYN", "INPP4A", "BACH1", "SERPINB9", "JAK3", "HLA-DPA1",
                "IGF2R", "TNFAIP8", "GRAMD1A", "B2M", "HLA-DPB1", "HLA-DRA", "IFNGR2")

selectGenes <- selectGenes[order(match(selectGenes,rownames(logFC_df_ser)))]

ha = rowAnnotation(genes = anno_mark(at = which(rownames(logFC_df_ser) %in% selectGenes), 
    labels = selectGenes, labels_gp = gpar(col = "black", fontfamily = "Arial", 
                                           fontface="italic", fontsize = 10)))

In [ ]:
%%R -w 5 -h 10 -r 300 --units in
# plot heatmap
set.seed(123)

hmap = Heatmap(name = "log2FC", logFC_df_ser, col = circlize::colorRamp2(breaks = c(-5, -2.5, 0, 2.5, 5),
                                                                colors=rev(RColorBrewer::brewer.pal(5, "RdBu"))),
                                                                #colors=viridis::viridis_pal(option = "RdBu")(5)),
        right_annotation = ha,
        show_row_names = FALSE,
        cluster_rows = F,
        cluster_columns = F,
        #row_names_gp = grid::gpar(fontsize = 2),
        heatmap_legend_param = list(direction = "horizontal",
                                   title_position = "topcenter"
                                   )
)

draw(hmap)

In [ ]:
%%R -w 5 -h 10 -r 300 --units in

set.seed(123)

hmap = Heatmap(name = "log2FC", logFC_df_ser, col = circlize::colorRamp2(breaks = c(-5, -2.5, 0, 2.5, 5),
                                                                colors=rev(RColorBrewer::brewer.pal(5, "RdBu"))),
                                                                #colors=viridis::viridis_pal(option = "RdBu")(5)),
        right_annotation = ha,
        show_row_names = FALSE,
        cluster_rows = F,
        cluster_columns = F,
        #row_names_gp = grid::gpar(fontsize = 2),
        heatmap_legend_param = list(
                                   title_position = "topleft",
                                at = c(-5, -2.5, 0, 2.5, 5),
                                labels = c('≤-5', '-2.5', '0', '2.5', '≥5')
                                   )
)

draw(hmap)

In [ ]:
%%R
# save aggregated edgeR results
saveRDS(edgeR_df, "../output/human/human_microglia_edgeR_DE_results_all.rds")
write.csv(edgeR_df, "../output/human/human_microglia_edgeR_DE_results_all.csv")
saveRDS(sig_degs_df, "../output/human/human_microglia_edgeR_significant_degs_all.rds")
write.csv(sig_degs_df, "../output/human/human_microglia_edgeR_significant_degs_all.csv")

In [ ]:
%%R

saveRDS(up_degs, "../output/human/human_microglia_edgeR_upregulated_degs_list.rds")
saveRDS(down_degs, "../output/human/human_microglia_edgeR_upregulated_degs_list.rds")
saveRDS(up_mat, "../output/human/human_microglia_edgeR_upregulated_degs_combinationmatrix.rds")
saveRDS(down_mat, "../output/human/human_microglia_edgeR_downregulated_degs_combinationmatrix.rds")

In [ ]:
# now, run GSEA enrichment analysis using MSigDB Hallmarks gene set

In [ ]:
# retrieve MSigDB gene sets
msig = Msigdb()
# retrieve mouse hallmark gene sets
gmt = msig.get_gmt(category='h.all', dbver="2023.2.Hs")

In [ ]:
# we'll use the log2 fold change calculated by edgeR, multiplied by -log10(FDR) as our rank statistic

# cluster 2
cluster2_gsea_df = tt_group2
cluster2_gsea_df['rank'] = cluster2_gsea_df['logFC']*-1*np.log10(cluster2_gsea_df['FDR'])
cluster2_rank = cluster2_gsea_df['rank'].sort_values(ascending = False)

# cluster 5
cluster5_gsea_df = tt_group5
cluster5_gsea_df['rank'] = cluster5_gsea_df['logFC']*-1*np.log10(cluster5_gsea_df['FDR'])
cluster5_rank = cluster5_gsea_df['rank'].sort_values(ascending = False)

# cluster 1
cluster1_gsea_df = tt_group1
cluster1_gsea_df['rank'] = cluster1_gsea_df['logFC']*-1*np.log10(cluster1_gsea_df['FDR'])
# sort gene list by rank statistic
cluster1_rank = cluster1_gsea_df['rank'].sort_values(ascending = False)

# cluster 6
cluster6_gsea_df = tt_group6
cluster6_gsea_df['rank'] = cluster6_gsea_df['logFC']*-1*np.log10(cluster6_gsea_df['FDR'])
# sort gene list by rank statistic
cluster6_rank = cluster6_gsea_df['rank'].sort_values(ascending = False)

# cluster 4
cluster4_gsea_df = tt_group4
cluster4_gsea_df['rank'] = cluster4_gsea_df['logFC']*-1*np.log10(cluster4_gsea_df['FDR'])
cluster4_rank = cluster4_gsea_df['rank'].sort_values(ascending = False)

In [ ]:
# run GSEA Prerank test <-  Cluster 2
cluster2_gsea_res = gseapy.prerank(rnk = cluster2_rank, 
        gene_sets=gmt,
        threads = 48,
        permutation_num=1000,
        outdir=None,
        verbose = True,
        seed=0)

cluster2_gsea_res_df = cluster2_gsea_res.res2d

# examine significant GSEA results
cluster2_gsea_res_df[(cluster2_gsea_res_df['FWER p-val'] < 0.05)]

In [ ]:
# create GSEA results plot
with rc_context({"figure.figsize": (5, 5), "grid.alpha":0}): 
    axs = cluster2_gsea_res.plot(terms=cluster2_gsea_res_df[(cluster2_gsea_res_df['FWER p-val'] < 0.05)].Term)

In [ ]:
# run GSEA Prerank test <-  Cluster 5
cluster5_gsea_res = gseapy.prerank(rnk = cluster5_rank, 
        gene_sets=gmt,
        threads = 48,
        permutation_num=1000,
        outdir=None,
        verbose = True,
        seed=0)

cluster5_gsea_res_df = cluster5_gsea_res.res2d

# examine significant GSEA results
cluster5_gsea_res_df[(cluster5_gsea_res_df['FWER p-val'] < 0.05)]

In [ ]:
# create GSEA results plot
with rc_context({"figure.figsize": (5, 5), "grid.alpha":0}): 
    axs = cluster5_gsea_res.plot(terms=cluster5_gsea_res_df[(cluster5_gsea_res_df['FWER p-val'] < 0.05)].Term)

In [ ]:
# run GSEA Prerank test <-  Cluster 1
cluster1_gsea_res = gseapy.prerank(rnk = cluster1_rank, 
        gene_sets=gmt,
        threads = 48,
        permutation_num=1000,
        outdir=None,
        verbose = True,
        seed=0)

cluster1_gsea_res_df = cluster1_gsea_res.res2d

# examine significant GSEA results
cluster1_gsea_res_df[(cluster1_gsea_res_df['FWER p-val'] < 0.05)]

In [ ]:
# create GSEA results plot
with rc_context({"figure.figsize": (5, 5), "grid.alpha":0}): 
    axs = cluster1_gsea_res.plot(terms=cluster1_gsea_res_df[(cluster1_gsea_res_df['FWER p-val'] < 0.05)].Term)

In [ ]:
# run GSEA Prerank test <-  Cluster 6
cluster6_gsea_res = gseapy.prerank(rnk = cluster6_rank, 
        gene_sets=gmt,
        threads = 48,
        permutation_num=1000,
        outdir=None,
        verbose = True,
        seed=0)

cluster6_gsea_res_df = cluster6_gsea_res.res2d

# examine significant GSEA results
cluster6_gsea_res_df[(cluster6_gsea_res_df['FWER p-val'] < 0.05)]

In [ ]:
# create GSEA results plot
with rc_context({"figure.figsize": (5, 5), "grid.alpha":0}): 
    axs = cluster6_gsea_res.plot(terms=cluster6_gsea_res_df[(cluster6_gsea_res_df['FWER p-val'] < 0.05)].Term)

In [ ]:
# run GSEA Prerank test <-  Cluster 4
cluster4_gsea_res = gseapy.prerank(rnk = cluster4_rank, 
        gene_sets=gmt,
        threads = 48,
        permutation_num=1000,
        outdir=None,
        verbose = True,
        seed=0)

cluster4_gsea_res_df = cluster4_gsea_res.res2d

# examine significant GSEA results
cluster4_gsea_res_df[(cluster4_gsea_res_df['FWER p-val'] < 0.05)]

In [ ]:
# create GSEA results plot
with rc_context({"figure.figsize": (5, 5), "grid.alpha":0}): 
    axs = cluster4_gsea_res.plot(terms=cluster4_gsea_res_df[(cluster4_gsea_res_df['FWER p-val'] < 0.05)].Term)

In [ ]:
# save GSEA results
cluster1_gsea_res_df.to_csv("../output/human/human_microglia_cluster1_v_cluster3_degs_gsea_results.csv")
cluster1_gsea_res_df.to_pickle("../output/human/human_microglia_cluster1_v_cluster3_degs_gsea_results.pkl")

cluster2_gsea_res_df.to_csv("../output/human/human_microglia_cluster2_v_cluster3_degs_gsea_results.csv")
cluster2_gsea_res_df.to_pickle("../output/human/human_microglia_cluster2_v_cluster3_degs_gsea_results.pkl")

cluster5_gsea_res_df.to_csv("../output/human/human_microglia_cluster5_v_cluster3_degs_gsea_results.csv")
cluster5_gsea_res_df.to_pickle("../output/human/human_microglia_cluster5_v_cluster3_degs_gsea_results.pkl")

cluster6_gsea_res_df.to_csv("../output/human/human_microglia_cluster6_v_cluster3_degs_gsea_results.csv")
cluster6_gsea_res_df.to_pickle("../output/human/human_microglia_cluster6_v_cluster3_degs_gsea_results.pkl")

cluster4_gsea_res_df.to_csv("../output/human/human_microglia_cluster4_v_cluster3_degs_gsea_results.csv")
cluster4_gsea_res_df.to_pickle("../output/human/human_microglia_cluster4_v_cluster3_degs_gsea_results.pkl")

In [ ]:
enriched_terms = [cluster2_gsea_res_df[cluster2_gsea_res_df['FWER p-val'] < 0.05].Term.tolist() +  
                  cluster5_gsea_res_df[cluster5_gsea_res_df['FWER p-val'] < 0.05].Term.tolist() + 
                  cluster1_gsea_res_df[cluster1_gsea_res_df['FWER p-val'] < 0.05].Term.tolist() + 
                  cluster6_gsea_res_df[cluster6_gsea_res_df['FWER p-val'] < 0.05].Term.tolist() + 
                  cluster4_gsea_res_df[cluster4_gsea_res_df['FWER p-val'] < 0.05].Term.tolist()
                 ]

enriched_terms = enriched_terms[0]
enriched_terms = np.unique(enriched_terms)
enriched_terms

In [ ]:
enriched_terms = list(enriched_terms)

In [ ]:
# aggreage GSEA results
gsea_results_dict = {
    'Cluster2':cluster2_gsea_res_df, 
    'Cluster5':cluster5_gsea_res_df,
    'Cluster1':cluster1_gsea_res_df,
    'Cluster6':cluster6_gsea_res_df,
    'Cluster4':cluster4_gsea_res_df,
               }
gsea_df = []

for key in gsea_results_dict:
    for y in list(range(0, len(enriched_terms))):
        gsea_df.append(
        {
            'Comparison': key,
            'Set': enriched_terms[y],
            'NES': gsea_results_dict[key][gsea_results_dict[key].Term == enriched_terms[y]]['NES'].values[0],
            'pval': gsea_results_dict[key][gsea_results_dict[key].Term == enriched_terms[y]]['FWER p-val'].values[0]
        }
    )

gsea_df = pd.DataFrame(gsea_df)

In [ ]:
%%R -i gsea_df
# load data frame in R
gsea.df <- gsea_df %>% mutate(pval = ifelse(pval == 0, 0.001, pval)) %>% mutate(score = NES*-log10(pval))

In [ ]:
%%R

gsea.scores = reshape2::acast(gsea.df, Set~Comparison, value.var = "score")

In [ ]:
%%R

sig_mat = reshape2::acast(gsea.df, Set~Comparison, value.var = "pval")

In [ ]:
%%R

library(latex2exp)

In [ ]:
%%R -w 6 -h 5.5 -r 300 --units in
# plot heatmap
hmap = Heatmap(gsea.scores, name = "Enrichment", 
        col = circlize::colorRamp2(breaks = c(-5, -2.5, 0, 2.5, 5), 
                                   colors=rev(RColorBrewer::brewer.pal(5, "RdBu"))),
        row_labels = TeX(c('GLYCOLYSIS',
                           'HYPOXIA', 
                           'IL2-STAT5 SIGNALING',
                           'INFLAMMATORY RESPONSE',
                           'INTERFERON-$\\gamma$ RESPONSE',
                           'MTORC1 SIGNALING',
                           'MYC TARGETS',  
                            'P53 PATHWAY',
                           'PI3K-AKT-MTOR SIGNALING',
                            'TNF SIGNALING VIA NF-$\\kappa$B',
                            'UNFOLDED PROTEIN RESPONSE')),
        cell_fun = function(j, i, x, y, w, h, fill){
            if(sig_mat[i, j] <= 0.001){
            	gb = textGrob("***")
            	gb_w = convertWidth(grobWidth(gb), "mm")
            	gb_h = convertHeight(grobHeight(gb), "mm")
            	grid.text("***", x, y - gb_h*0.9 + gb_w*0.4)
            } else if(sig_mat[i, j] < 0.01){
            	gb = textGrob("**")
            	gb_w = convertWidth(grobWidth(gb), "mm")
            	gb_h = convertHeight(grobHeight(gb), "mm")
            	grid.text("**", x, y - gb_h*0.7 + gb_w*0.4)
            } else if(sig_mat[i, j] < 0.05){
            	gb = textGrob("*")
            	gb_w = convertWidth(grobWidth(gb), "mm")
            	gb_h = convertHeight(grobHeight(gb), "mm")
            	grid.text("*", x, y - gb_h*0.5 + gb_w*0.4)
            } else {
                gb = textGrob("")
            	gb_w = convertWidth(grobWidth(gb), "mm")
            	gb_h = convertHeight(grobHeight(gb), "mm")
                grid.text("", x, y - gb_h*0.5 + gb_w*0.4)
            }
        },
        #column_title = "Astrocyte Cluster", 
        row_title = "MSigDB Hallmark Gene Set",
        row_title_gp = grid::gpar(fontsize = 12),
        row_names_gp = grid::gpar(fontsize = 10),
        heatmap_legend_param = list(direction = "horizontal",
                                   title_position = "topcenter",
                                    at = c(-5, 0, 5),
                                labels = c('≤-5', '0', '≥5')
                                   )
       )

draw(hmap)

In [ ]:
%%R

# save aggregated results
saveRDS(gsea.df, "../output/human/human_microglia_msigdb_hallmark_gsea_results.rds")
write.csv(gsea.df, "../output/human/human_microglia_msigdb_hallmark_gsea_results.csv")

In [ ]:
# next we examine enrichment of previously described microglia states and macrophage activation states

In [ ]:
from mousipy import *

In [ ]:
### creating gene sets for major previously defined microglia states:

## Human states
# HAM (human AD microglia; human; Srinivasan et al 2020 [PMID: 32610143])
# MIMS (microglia inflamed in MS; MS; human; Absinta et al 2021 [PMID: 34497421])

## Mouse states
# DAM microglia (disease-associated microglia; mouse; Keren-Shaul et al 2017 [PMID: 28602351])
# LDAM (lipid droplet accumulating microglia; mouse; Marschallinger et al 2020 [PMID: 31959936])
# MGnD (microglial neurodegenerative phenotype; common to AD/ALS/EAE/aging; mouse; Krasemann et al 2018 [PMID: 28930663])

### leaving these out now
### also creating gene sets for specific cytokine-induced Macrophage polarization states defined in Cui et al 2023 [PMID: 38057668]
# Mac-a: Type I interferon induced;
# Mac-b: IFN gamma induced;
# Mac-c: IL-1a, IL-1b, IL-36a induced;
# Mac-d: TNF induced;
# Mac-e: IL-4; IL-14 induced;

In [ ]:
m2h_tab = pd.read_csv('~/mambaforge/envs/CART_analysis_2024/lib/python3.9/site-packages/mousipy/biomart/mouse_to_human.csv').set_index('Gene name')

In [ ]:
### Obtaining 'HAM' microglia gene set

In [ ]:
%%bash
# obtain HAM differential expression results table from Srinivasan et al 2020 [PMID: 32610143] (Supplementary Tables Data S2)
wget -O ../output/human/human_HAM_microglia_geneset.xlsx https://ars.els-cdn.com/content/image/1-s2.0-S221112472030824X-mmc3.xlsx

In [ ]:
ham = pd.read_excel("../output/human/human_HAM_microglia_geneset.xlsx", header = 17, usecols = "M,EC:EE")

In [ ]:
ham_genes = ham[(ham['Log2FC.2'] > 0) & (ham['adjP.2'] < 0.05)]['Unnamed: 12'].values

In [ ]:
ham_genes = [x for x in ham_genes if str(x) != 'nan']

In [ ]:
### Obtaining 'MIMS' microglia gene sets

In [ ]:
%%bash
# obtain MIMS differential expression results table from Absinta et al 2021 [PMID: 34497421] (Supplementary Tables)
wget -O ../output/human/human_mims_microglia_genesets.xlsx https://www.ncbi.nlm.nih.gov/pmc/articles/PMC8719282/bin/NIHMS1762170-supplement-Supplementary_Tables.xlsx

In [ ]:
mims = pd.read_excel("../output/human/human_mims_microglia_genesets.xlsx", sheet_name = "Table S4_IMM", header = 1)

In [ ]:
mimsFE_genes = mims[(mims['cluster'] == 8) & (mims['avg_logFC'] > 0.5) & (mims['p_val_adj'] < 0.05)].gene.values
mimsFOAM_genes = mims[(mims['cluster'] == 1) & (mims['avg_logFC'] > 0.5) & (mims['p_val_adj'] < 0.05)].gene.values

In [ ]:
### Obtaining 'DAM' microglia gene set

In [ ]:
%%bash
# obtain DAM versus homeostatic microglia differential expression results table from Keren-Shaul et al 2017 [PMID: 28602351] (Supplementary Table 3)
wget -O ../output/human/mouse_DAM_microglia_geneset.xlsx https://ars.els-cdn.com/content/image/1-s2.0-S0092867417305780-mmc3.xlsx

In [ ]:
dam = pd.read_excel("../output/human/mouse_DAM_microglia_geneset.xlsx")

In [ ]:
dam_genes = dam[(dam['Fold-change (DAM to homeostatic microglia)'] > 0.5) & (dam['DAM FDR p-value'] > -1*np.log10(0.00001))]['Gene name'].values

In [ ]:
dam_genes = check_orthologs(dam_genes, target='Symbol')[0]
dam_genes_HUMAN = list(dam_genes.values())

In [ ]:
### Obtaining 'LDAM' microglia gene set

In [ ]:
%%bash
# obtain LDAM differential expression results table from Marschallinger et al 2020 [PMID: 31959936] (Supplementary Table 2)
wget -O ../output/human/mouse_LDAM_microglia_geneset.xlsx https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7595134/bin/NIHMS1544918-supplement-1.xlsx

In [ ]:
ldam = pd.read_excel("../output/human/mouse_LDAM_microglia_geneset.xlsx", sheet_name = "T2 1- RNA Seq Aging")

In [ ]:
ldam_genes = ldam[(ldam['log2FoldChange'] > 0.25) & (ldam['padj'] < 0.05)]['Unnamed: 0'].values

In [ ]:
ldam_genes = check_orthologs(ldam_genes, target='Symbol')[0]
ldam_genes_HUMAN = list(ldam_genes.values())

In [ ]:
### Obtaining MGnD signature gene set

In [ ]:
%%bash
# obtain MGnD signature genes table from Keren-Shaul et al 2017 [PMID: 28602351] (Supplementary Table 3)
wget -O ../output/human/mouse_MGnD_microglia_geneset.xlsx https://www.ncbi.nlm.nih.gov/pmc/articles/PMC5719893/bin/NIHMS901481-supplement-2.xlsx

In [ ]:
mgnd = pd.read_excel("../output/human/mouse_MGnD_microglia_geneset.xlsx", sheet_name = "Common afected genes")

In [ ]:
mgnd_genes = mgnd[0:28]['Common affected genes in disease'] # only the first 28 genes are upregulated in MGnD signature

In [ ]:
mgnd_genes = mgnd_genes.values

In [ ]:
mgnd_genes = check_orthologs(mgnd_genes, target='Symbol')[0]
mgnd_genes_HUMAN = list(mgnd_genes.values())

In [ ]:
### Obtaining ImmuneDictionary gene sets from Cui et al 2023 [PMID: 38057668]

In [ ]:
%%bash
# obtain immune dictionary macrophage polarization gene sets (single-cell RNA-seq differential expression results)
wget -O ../output/human/mouse_immune_dictionary_polarization_genesets.xlsx https://static-content.springer.com/esm/art%3A10.1038%2Fs41586-023-06816-9/MediaObjects/41586_2023_6816_MOESM9_ESM.xlsx

In [ ]:
immunedict = pd.read_excel("../output/human/mouse_immune_dictionary_polarization_genesets.xlsx", sheet_name = "Macrophage")

In [ ]:
# examine gene set sizes for each macropahge polarization state using stricter significance cutoffs
immunedict[(immunedict.Avg_log2FC > 0) & (immunedict.P_adj < 0.05)]['Polarization'].value_counts()

In [ ]:
# extract genes exceeding threshold
immunedict = immunedict[(immunedict.Avg_log2FC > 0) & (immunedict.P_adj < 0.05)]

In [ ]:
# create gene set lists
mac_a = immunedict[immunedict['Polarization'] == "Mac-a"].Gene.values
mac_b = immunedict[immunedict['Polarization'] == "Mac-b"].Gene.values
mac_c = immunedict[immunedict['Polarization'] == "Mac-c"].Gene.values
mac_d = immunedict[immunedict['Polarization'] == "Mac-d"].Gene.values
mac_e = immunedict[immunedict['Polarization'] == "Mac-e"].Gene.values

In [ ]:
# convert to human gene symbols
mac_a_HUMAN = check_orthologs(mac_a, target='Symbol')[0]
mac_a_HUMAN = list(mac_a_HUMAN.values())

mac_b_HUMAN = check_orthologs(mac_b, target='Symbol')[0]
mac_b_HUMAN = list(mac_b_HUMAN.values())

mac_c_HUMAN = check_orthologs(mac_c, target='Symbol')[0]
mac_c_HUMAN = list(mac_c_HUMAN.values())

mac_d_HUMAN = check_orthologs(mac_d, target='Symbol')[0]
mac_d_HUMAN = list(mac_d_HUMAN.values())

mac_e_HUMAN = check_orthologs(mac_e, target='Symbol')[0]
mac_e_HUMAN = list(mac_e_HUMAN.values())

In [ ]:
# create dictionary of gene sets
mg_genesets = {
    'HAM':ham_genes,
    'MIMS-iron':list(mimsFE_genes),
    'MIMS-foamy':list(mimsFOAM_genes),
    'DAM':dam_genes_HUMAN,
    'LDAM':ldam_genes_HUMAN,
    'MGND':mgnd_genes_HUMAN,
    'Mac_Polarization_A':mac_a_HUMAN,
    'Mac_Polarization_B':mac_b_HUMAN,
    'Mac_Polarization_C':mac_c_HUMAN,
    'Mac_Polarization_D':mac_d_HUMAN,
    'Mac_Polarization_E':mac_e_HUMAN
}

In [ ]:
import csv
import itertools

def save_dict_to_csv(data_dict, filename):
    # Get the maximum length of the arrays in the dictionary
    max_length = max(len(v) for v in data_dict.values())
    
    # Prepare the data for CSV writing
    rows = []
    for i in range(max_length):
        row = []
        for key in data_dict.keys():
            try:
                row.append(data_dict[key][i])
            except IndexError:
                row.append('')  # Fill with empty string if index is out of range
        rows.append(row)

    # Write the data to CSV file
    with open(filename, mode='w', newline='') as file:
        writer = csv.writer(file)
        # Write the header
        writer.writerow(data_dict.keys())
        # Write the rows
        writer.writerows(rows)

In [ ]:
# save gene set dictionary as csv file
save_dict_to_csv(mg_genesets, '../output/human/human_microglia_mgstate_genesets.csv')

In [ ]:
# also save as npy file
np.save('../output/human/human_microglia_states_genesets.npy', mg_genesets) 

In [ ]:
# define radar chart for plotting MG state enrichment

from matplotlib.patches import Circle, RegularPolygon
from matplotlib.path import Path
from matplotlib.projections import register_projection
from matplotlib.projections.polar import PolarAxes
from matplotlib.spines import Spine
from matplotlib.transforms import Affine2D


def radar_factory(num_vars, frame='circle'):
    """
    Create a radar chart with `num_vars` Axes.

    This function creates a RadarAxes projection and registers it.

    Parameters
    ----------
    num_vars : int
        Number of variables for radar chart.
    frame : {'circle', 'polygon'}
        Shape of frame surrounding Axes.

    """
    # calculate evenly-spaced axis angles
    theta = np.linspace(0, 2*np.pi, num_vars, endpoint=False)

    class RadarTransform(PolarAxes.PolarTransform):

        def transform_path_non_affine(self, path):
            # Paths with non-unit interpolation steps correspond to gridlines,
            # in which case we force interpolation (to defeat PolarTransform's
            # autoconversion to circular arcs).
            if path._interpolation_steps > 1:
                path = path.interpolated(num_vars)
            return Path(self.transform(path.vertices), path.codes)

    class RadarAxes(PolarAxes):

        name = 'radar'
        PolarTransform = RadarTransform

        def __init__(self, *args, **kwargs):
            super().__init__(*args, **kwargs)
            # rotate plot such that the first axis is at the top
            self.set_theta_zero_location('N')

        def fill(self, *args, closed=True, **kwargs):
            """Override fill so that line is closed by default"""
            return super().fill(closed=closed, *args, **kwargs)

        def plot(self, *args, **kwargs):
            """Override plot so that line is closed by default"""
            lines = super().plot(*args, **kwargs)
            for line in lines:
                self._close_line(line)

        def _close_line(self, line):
            x, y = line.get_data()
            # FIXME: markers at x[0], y[0] get doubled-up
            if x[0] != x[-1]:
                x = np.append(x, x[0])
                y = np.append(y, y[0])
                line.set_data(x, y)

        def set_varlabels(self, labels):
            self.set_thetagrids(np.degrees(theta), labels)
            
        def _gen_axes_patch(self):
            # The Axes patch must be centered at (0.5, 0.5) and of radius 0.5
            # in axes coordinates.
            if frame == 'circle':
                return Circle((0.5, 0.5), 0.5)
            elif frame == 'polygon':
                return RegularPolygon((0.5, 0.5), num_vars,
                                      radius=.5, edgecolor="k")
            else:
                raise ValueError("Unknown value for 'frame': %s" % frame)

        def _gen_axes_spines(self):
            if frame == 'circle':
                return super()._gen_axes_spines()
            elif frame == 'polygon':
                # spine_type must be 'left'/'right'/'top'/'bottom'/'circle'.
                spine = Spine(axes=self,
                              spine_type='circle',
                              path=Path.unit_regular_polygon(num_vars))
                # unit_regular_polygon gives a polygon of radius 1 centered at
                # (0, 0) but we want a polygon of radius 0.5 centered at (0.5,
                # 0.5) in axes coordinates.
                spine.set_transform(Affine2D().scale(.5).translate(.5, .5)
                                    + self.transAxes)
                return {'polar': spine}
            else:
                raise ValueError("Unknown value for 'frame': %s" % frame)

    register_projection(RadarAxes)
    return theta

In [ ]:
from gseapy import gseaplot2
from matplotlib.colors import ListedColormap
import matplotlib.colors
import matplotlib.ticker as ticker

In [ ]:
# define functions for plotting combined MG state GSEA plots

def extract_colors_hexcodes(n, cmap_name):
    # Get the default colormap
    default_cmap = plt.get_cmap(cmap_name)
    
    # Extract n discrete colors from the colormap
    cmap_colors = default_cmap(np.linspace(0, 1, n))
    
    # Convert RGB colors to hexcodes
    hexcodes = [mpl.colors.rgb2hex(color) for color in cmap_colors]

    return hexcodes


def adjust_opacity(hexcodes, gsea_res_df):
    # Define opacity values based on the associated data frame column

    alphas = []
    
    for x in np.arange(len(hexcodes)):
        if gsea_res_df.iloc[::-1][gsea_res_df.index == x]['FWER p-val'].values[0] < 0.05:
            alphas.append(1)
        else:
            alphas.append(0.35)
    
    # Adjust opacity of hexcodes based on associated values
    adjusted_hexcodes = []
    for hexcode, alpha in zip(hexcodes, alphas):
        foreground_tuple = mpl.colors.hex2color(hexcode) + (alpha,)
        foreground_arr = np.array(foreground_tuple)
        rgba_color = tuple( (1. -  alpha) + foreground_arr*alpha )
        adjusted_hexcode = mpl.colors.rgb2hex(rgba_color)
        adjusted_hexcodes.append(adjusted_hexcode)
    
    return adjusted_hexcodes

def reorder_hexcodes(term_list, categories_df):
    # Merge the hexcodes and categories DataFrame on the category_column
    merged_df = pd.DataFrame({'Term': term_list})
    merged_df = pd.merge(merged_df, categories_df, on='Term', how = "left")
    
    # Get the sorted hexcodes
    sorted_hexcodes = merged_df['hexcode'].tolist()
    
    return sorted_hexcodes

In [ ]:
# run GSEA Prerank test <-  Cluster 2
cluster2_mgstate_res = gseapy.prerank(rnk = cluster2_rank, 
        gene_sets=mg_genesets,
        threads = 48,
        permutation_num=1000,
        outdir=None,
        verbose = True,
        seed=0)

cluster2_mgstate_res_df = cluster2_mgstate_res.res2d

# examine significant GSEA results
cluster2_mgstate_res_df[(cluster2_mgstate_res_df['FWER p-val'] < 0.05)]

In [ ]:
pal = extract_colors_hexcodes(8, 'turbo')
categories = ['DAM', 'LDAM', 'MIMS-iron', 'MIMS-foamy', 'Mac_Polarization_A', 'Mac_Polarization_B', 'Mac_Polarization_C', 'Mac_Polarization_D']

categories_df = pd.DataFrame({'Term': categories, 'hexcode':pal[::-1]})

terms = cluster2_mgstate_res_df.iloc[::-1].Term
hits = [cluster2_mgstate_res.results[t]['hits'] for t in terms]
runes = [cluster2_mgstate_res.results[t]['RES'] for t in terms]

with rc_context({"figure.figsize": (3, 4), "grid.alpha":0}): 
    ax = gseaplot2(terms=terms, RESs=runes, hits=hits, 
                   rank_metric=cluster2_mgstate_res.ranking, 
                   colors = adjust_opacity(reorder_hexcodes(terms, categories_df), cluster2_mgstate_res_df),
                   figsize=(3,4))

    ax[8].set_title('MG-A', fontsize = 20, fontweight="bold")
    ax[0].set_xlabel('Gene Rank', fontsize = 13)
    ax[9].tick_params(axis="both", which="both", right=True, labelsize = 8)
    ax[8].set_ylabel('Enrichment Score', fontsize = 13)
    ax[9].set_ylabel('Rank Metric', fontsize = 13)
    ax[8].tick_params(axis="both", which="both", left=True, labelsize = 10)
    ax[8].get_legend().remove()

    terms.reset_index(drop=True, inplace=True)

    for x in terms.index.values:
        
        if terms[terms.index.values == x].values[0] == "Mac_Polarization_A":
            name = "IFN-I polarized"
        elif terms[terms.index.values == x].values[0] == "Mac_Polarization_B":
            name = "IFN-γ polarized"
        elif terms[terms.index.values == x].values[0] == "Mac_Polarization_C":
            name = "IL-1 polarized"
        elif terms[terms.index.values == x].values[0] == "Mac_Polarization_D":
            name = "TNF polarized"
        else:
            name = terms[terms.index.values == x].values[0]
        ax2 = ax[x].twinx()
        ax[x].tick_params(
            axis="both", which="both", bottom=False, top=False, right=True, left=True, labelleft = True, color = "white", labelsize = 10
        )

        if cluster2_mgstate_res_df[::-1][cluster2_mgstate_res_df.index == x]['FWER p-val'].values[0] == 0:
            pval = 0.001
            pval_annotation = '         p<'+str(pval)
        elif cluster2_mgstate_res_df[::-1][cluster2_mgstate_res_df.index == x]['FWER p-val'].values[0] < 0.01:
            pval = round(cluster2_mgstate_res_df[::-1][cluster2_mgstate_res_df.index == x]['FWER p-val'].values[0], 3)
            pval_annotation = '         p='+str(pval)
        else: 
            pval = round(cluster2_mgstate_res_df[::-1][cluster2_mgstate_res_df.index == x]['FWER p-val'].values[0], 2)
            pval_annotation = '         p='+str(pval)

        tick_spacing = 0.5
        ax2.yaxis.set_minor_locator(ticker.MultipleLocator(tick_spacing))
        ax2.tick_params(axis='y', which='major', right = False, labelright = False)
        ax2.tick_params(axis='y', which='minor', tick1On=False, tick2On=False)
        
        ax[x].set_yticks(ax[x].get_yticks())
        ax[x].set_yticklabels(labels = ['', '', name, '', ''])
        ax2.set_ylabel(pval_annotation, rotation = 0, va='center', size = 10)


In [ ]:
state_names = ['DAM', 'MIMS-iron', 'MIMS-foamy', 'LDAM', 'Mac_Polarization_A', 'Mac_Polarization_B', 'Mac_Polarization_C', 'Mac_Polarization_D']

pvals = []
for x in state_names:
    val = cluster2_mgstate_res_df[cluster2_mgstate_res_df.Term == x]['FWER p-val'].values[0]
    if val == 0:
        val = 0.001
    val = -1*np.log10(val)
    pvals.append(val)

data = [['DAM', 'MSi', 'MSf', 'LDAM', 'A', 'B', 'C', 'D'], 
        ('MG-A', [
            pvals
        ])]

with rc_context({"figure.figsize": (4, 4)}): 
    N = len(data[0])
    theta = radar_factory(N, frame='polygon')
    
    spoke_labels = data.pop(0)
    title, case_data = data[0]
    
    fig, ax = plt.subplots(figsize=(4, 4), subplot_kw=dict(projection='radar'))
    fig.subplots_adjust(top=0.85, bottom=0.05)

    ax.set_rgrids([-1*np.log10(0.05), -1*np.log10(0.01), -1*np.log10(0.001)], labels = ["", "", ""], size = 10)
    ax.set_title(title,  position=(0.5, 1.1), ha='center', size = 20)

    ax.yaxis.grid(linestyle="dashed", alpha = 1, zorder = 1)
    ax.xaxis.grid(alpha = 0.1)
    ax.scatter(theta, case_data, s=20, c='indigo', zorder=10)
    
    for d in case_data:
        line = ax.plot(theta, d, color = "indigo", zorder = 5)
        ax.fill(theta, d,  alpha=0.75, color = "indigo", zorder = 4)
    ax.set_varlabels(spoke_labels)
    
    plt.show()

In [ ]:
# run GSEA Prerank test <-  Cluster 5
cluster5_mgstate_res = gseapy.prerank(rnk = cluster5_rank, 
        gene_sets=mg_genesets,
        threads = 48,
        permutation_num=1000,
        outdir=None,
        verbose = True,
        seed=0)

cluster5_mgstate_res_df = cluster5_mgstate_res.res2d

# examine significant GSEA results
cluster5_mgstate_res_df[(cluster5_mgstate_res_df['FWER p-val'] < 0.05)]

In [ ]:
pal = extract_colors_hexcodes(8, 'turbo')
categories = ['DAM', 'LDAM', 'MIMS-iron', 'MIMS-foamy', 'Mac_Polarization_A', 'Mac_Polarization_B', 'Mac_Polarization_C', 'Mac_Polarization_D']

categories_df = pd.DataFrame({'Term': categories, 'hexcode':pal[::-1]})

terms = cluster5_mgstate_res_df.iloc[::-1].Term
hits = [cluster5_mgstate_res.results[t]['hits'] for t in terms]
runes = [cluster5_mgstate_res.results[t]['RES'] for t in terms]

with rc_context({"figure.figsize": (3, 4), "grid.alpha":0}): 
    ax = gseaplot2(terms=terms, RESs=runes, hits=hits, 
                   rank_metric=cluster5_mgstate_res.ranking, 
                   colors = adjust_opacity(reorder_hexcodes(terms, categories_df), cluster5_mgstate_res_df),
                   figsize=(3,4))

    ax[8].set_title('MG-B', fontsize = 20, fontweight="bold")
    ax[0].set_xlabel('Gene Rank', fontsize = 13)
    ax[9].tick_params(axis="both", which="both", right=True, labelsize = 8)
    ax[8].set_ylabel('Enrichment Score', fontsize = 13)
    ax[9].set_ylabel('Rank Metric', fontsize = 13)
    ax[8].tick_params(axis="both", which="both", left=True, labelsize = 10)
    ax[8].get_legend().remove()

    terms.reset_index(drop=True, inplace=True)

    for x in terms.index.values:
        
        if terms[terms.index.values == x].values[0] == "Mac_Polarization_A":
            name = "IFN-I polarized"
        elif terms[terms.index.values == x].values[0] == "Mac_Polarization_B":
            name = "IFN-γ polarized"
        elif terms[terms.index.values == x].values[0] == "Mac_Polarization_C":
            name = "IL-1 polarized"
        elif terms[terms.index.values == x].values[0] == "Mac_Polarization_D":
            name = "TNF polarized"
        else:
            name = terms[terms.index.values == x].values[0]
        ax2 = ax[x].twinx()
        ax[x].tick_params(
            axis="both", which="both", bottom=False, top=False, right=True, left=True, labelleft = True, color = "white", labelsize = 10
        )

        if cluster5_mgstate_res_df[::-1][cluster5_mgstate_res_df.index == x]['FWER p-val'].values[0] == 0:
            pval = 0.001
            pval_annotation = '         p<'+str(pval)
        elif cluster5_mgstate_res_df[::-1][cluster5_mgstate_res_df.index == x]['FWER p-val'].values[0] < 0.01:
            pval = round(cluster5_mgstate_res_df[::-1][cluster5_mgstate_res_df.index == x]['FWER p-val'].values[0], 3)
            pval_annotation = '         p='+str(pval)
        else: 
            pval = round(cluster5_mgstate_res_df[::-1][cluster5_mgstate_res_df.index == x]['FWER p-val'].values[0], 2)
            pval_annotation = '         p='+str(pval)

        tick_spacing = 0.5
        ax2.yaxis.set_minor_locator(ticker.MultipleLocator(tick_spacing))
        ax2.tick_params(axis='y', which='major', right = False, labelright = False)
        ax2.tick_params(axis='y', which='minor', tick1On=False, tick2On=False)
        
        ax[x].set_yticks(ax[x].get_yticks())
        ax[x].set_yticklabels(labels = ['', '', name, '', ''])
        ax2.set_ylabel(pval_annotation, rotation = 0, va='center', size = 10)


In [ ]:
state_names = ['DAM', 'MIMS-iron', 'MIMS-foamy', 'LDAM', 'Mac_Polarization_A', 'Mac_Polarization_B', 'Mac_Polarization_C', 'Mac_Polarization_D']

pvals = []
for x in state_names:
    val = cluster5_mgstate_res_df[cluster5_mgstate_res_df.Term == x]['FWER p-val'].values[0]
    if val == 0:
        val = 0.001
    val = -1*np.log10(val)
    pvals.append(val)

data = [['DAM', 'MSi', 'MSf', 'LDAM', 'A', 'B', 'C', 'D'], 
        ('MG-B', [
            pvals
        ])]

with rc_context({"figure.figsize": (4, 4)}): 
    N = len(data[0])
    theta = radar_factory(N, frame='polygon')
    
    spoke_labels = data.pop(0)
    title, case_data = data[0]
    
    fig, ax = plt.subplots(figsize=(4, 4), subplot_kw=dict(projection='radar'))
    fig.subplots_adjust(top=0.85, bottom=0.05)

    ax.set_rgrids([-1*np.log10(0.05), -1*np.log10(0.01), -1*np.log10(0.001)], labels = ["", "", ""], size = 10)
    ax.set_title(title,  position=(0.5, 1.1), ha='center', size = 20)

    ax.yaxis.grid(linestyle="dashed", alpha = 1, zorder = 1)
    ax.xaxis.grid(alpha = 0.1)
    ax.scatter(theta, case_data, s=20, c='violet', zorder=10)
    
    for d in case_data:
        line = ax.plot(theta, d, color = "violet", zorder = 5)
        ax.fill(theta, d,  alpha=0.75, color = "violet", zorder = 4)
    ax.set_varlabels(spoke_labels)
    
    plt.show()

In [ ]:
# run GSEA Prerank test <-  Cluster 1
cluster1_mgstate_res = gseapy.prerank(rnk = cluster1_rank, 
        gene_sets=mg_genesets,
        threads = 48,
        permutation_num=1000,
        outdir=None,
        verbose = True,
        seed=0)

cluster1_mgstate_res_df = cluster1_mgstate_res.res2d

# examine significant GSEA results
cluster1_mgstate_res_df[(cluster1_mgstate_res_df['FWER p-val'] < 0.05)]

In [ ]:
pal = extract_colors_hexcodes(8, 'turbo')
categories = ['DAM', 'LDAM', 'MIMS-iron', 'MIMS-foamy', 'Mac_Polarization_A', 'Mac_Polarization_B', 'Mac_Polarization_C', 'Mac_Polarization_D']

categories_df = pd.DataFrame({'Term': categories, 'hexcode':pal[::-1]})

terms = cluster1_mgstate_res_df.iloc[::-1].Term
hits = [cluster1_mgstate_res.results[t]['hits'] for t in terms]
runes = [cluster1_mgstate_res.results[t]['RES'] for t in terms]

with rc_context({"figure.figsize": (3, 4), "grid.alpha":0}): 
    ax = gseaplot2(terms=terms, RESs=runes, hits=hits, 
                   rank_metric=cluster1_mgstate_res.ranking, 
                   colors = adjust_opacity(reorder_hexcodes(terms, categories_df), cluster1_mgstate_res_df),
                   figsize=(3,4))

    ax[8].set_title('MG-C1', fontsize = 20, fontweight="bold")
    ax[0].set_xlabel('Gene Rank', fontsize = 13)
    ax[9].tick_params(axis="both", which="both", right=True, labelsize = 8)
    ax[8].set_ylabel('Enrichment Score', fontsize = 13)
    ax[9].set_ylabel('Rank Metric', fontsize = 13)
    ax[8].tick_params(axis="both", which="both", left=True, labelsize = 10)
    ax[8].get_legend().remove()

    terms.reset_index(drop=True, inplace=True)

    for x in terms.index.values:
        
        if terms[terms.index.values == x].values[0] == "Mac_Polarization_A":
            name = "IFN-I polarized"
        elif terms[terms.index.values == x].values[0] == "Mac_Polarization_B":
            name = "IFN-γ polarized"
        elif terms[terms.index.values == x].values[0] == "Mac_Polarization_C":
            name = "IL-1 polarized"
        elif terms[terms.index.values == x].values[0] == "Mac_Polarization_D":
            name = "TNF polarized"
        else:
            name = terms[terms.index.values == x].values[0]
        ax2 = ax[x].twinx()
        ax[x].tick_params(
            axis="both", which="both", bottom=False, top=False, right=True, left=True, labelleft = True, color = "white", labelsize = 10
        )

        if cluster1_mgstate_res_df[::-1][cluster1_mgstate_res_df.index == x]['FWER p-val'].values[0] == 0:
            pval = 0.001
            pval_annotation = '         p<'+str(pval)
        elif cluster1_mgstate_res_df[::-1][cluster1_mgstate_res_df.index == x]['FWER p-val'].values[0] < 0.01:
            pval = round(cluster1_mgstate_res_df[::-1][cluster1_mgstate_res_df.index == x]['FWER p-val'].values[0], 3)
            pval_annotation = '         p='+str(pval)
        else: 
            pval = round(cluster1_mgstate_res_df[::-1][cluster1_mgstate_res_df.index == x]['FWER p-val'].values[0], 2)
            pval_annotation = '         p='+str(pval)

        tick_spacing = 0.5
        ax2.yaxis.set_minor_locator(ticker.MultipleLocator(tick_spacing))
        ax2.tick_params(axis='y', which='major', right = False, labelright = False)
        ax2.tick_params(axis='y', which='minor', tick1On=False, tick2On=False)
        
        ax[x].set_yticks(ax[x].get_yticks())
        ax[x].set_yticklabels(labels = ['', '', name, '', ''])
        ax2.set_ylabel(pval_annotation, rotation = 0, va='center', size = 10)


In [ ]:
state_names = ['DAM', 'MIMS-iron', 'MIMS-foamy', 'LDAM', 'Mac_Polarization_A', 'Mac_Polarization_B', 'Mac_Polarization_C', 'Mac_Polarization_D']

pvals = []
for x in state_names:
    val = cluster1_mgstate_res_df[cluster1_mgstate_res_df.Term == x]['FWER p-val'].values[0]
    if val == 0:
        val = 0.001
    val = -1*np.log10(val)
    pvals.append(val)

data = [['DAM', 'MSi', 'MSf', 'LDAM', 'A', 'B', 'C', 'D'], 
        ('MG-C1', [
            pvals
        ])]

with rc_context({"figure.figsize": (4, 4)}): 
    N = len(data[0])
    theta = radar_factory(N, frame='polygon')
    
    spoke_labels = data.pop(0)
    title, case_data = data[0]
    
    fig, ax = plt.subplots(figsize=(4, 4), subplot_kw=dict(projection='radar'))
    fig.subplots_adjust(top=0.85, bottom=0.05)

    ax.set_rgrids([-1*np.log10(0.05), -1*np.log10(0.01), -1*np.log10(0.001)], labels = ["", "", ""], size = 10)
    ax.set_title(title,  position=(0.5, 1.1), ha='center', size = 20)

    ax.yaxis.grid(linestyle="dashed", alpha = 1, zorder = 1)
    ax.xaxis.grid(alpha = 0.1)
    ax.scatter(theta, case_data, s=20, c='maroon', zorder=10)
    
    for d in case_data:
        line = ax.plot(theta, d, color = "maroon", zorder = 5)
        ax.fill(theta, d,  alpha=0.75, color = "maroon", zorder = 4)
    ax.set_varlabels(spoke_labels)
    
    plt.show()

In [ ]:
# run GSEA Prerank test <-  Cluster 6
cluster6_mgstate_res = gseapy.prerank(rnk = cluster6_rank, 
        gene_sets=mg_genesets,
        threads = 48,
        permutation_num=1000,
        outdir=None,
        verbose = True,
        seed=0)

cluster6_mgstate_res_df = cluster6_mgstate_res.res2d

# examine significant GSEA results
cluster6_mgstate_res_df[(cluster6_mgstate_res_df['FWER p-val'] < 0.05)]

In [ ]:
pal = extract_colors_hexcodes(8, 'turbo')
categories = ['DAM', 'LDAM', 'MIMS-iron', 'MIMS-foamy', 'Mac_Polarization_A', 'Mac_Polarization_B', 'Mac_Polarization_C', 'Mac_Polarization_D']

categories_df = pd.DataFrame({'Term': categories, 'hexcode':pal[::-1]})

terms = cluster6_mgstate_res_df.iloc[::-1].Term
hits = [cluster6_mgstate_res.results[t]['hits'] for t in terms]
runes = [cluster6_mgstate_res.results[t]['RES'] for t in terms]

with rc_context({"figure.figsize": (3, 4), "grid.alpha":0}): 
    ax = gseaplot2(terms=terms, RESs=runes, hits=hits, 
                   rank_metric=cluster6_mgstate_res.ranking, 
                   colors = adjust_opacity(reorder_hexcodes(terms, categories_df), cluster6_mgstate_res_df),
                   figsize=(3,4))

    ax[8].set_title('MG-C2', fontsize = 20, fontweight="bold")
    ax[0].set_xlabel('Gene Rank', fontsize = 13)
    ax[9].tick_params(axis="both", which="both", right=True, labelsize = 8)
    ax[8].set_ylabel('Enrichment Score', fontsize = 13)
    ax[9].set_ylabel('Rank Metric', fontsize = 13)
    ax[8].tick_params(axis="both", which="both", left=True, labelsize = 10)
    ax[8].get_legend().remove()

    terms.reset_index(drop=True, inplace=True)

    for x in terms.index.values:
        
        if terms[terms.index.values == x].values[0] == "Mac_Polarization_A":
            name = "IFN-I polarized"
        elif terms[terms.index.values == x].values[0] == "Mac_Polarization_B":
            name = "IFN-γ polarized"
        elif terms[terms.index.values == x].values[0] == "Mac_Polarization_C":
            name = "IL-1 polarized"
        elif terms[terms.index.values == x].values[0] == "Mac_Polarization_D":
            name = "TNF polarized"
        else:
            name = terms[terms.index.values == x].values[0]
        ax2 = ax[x].twinx()
        ax[x].tick_params(
            axis="both", which="both", bottom=False, top=False, right=True, left=True, labelleft = True, color = "white", labelsize = 10
        )

        if cluster6_mgstate_res_df[::-1][cluster6_mgstate_res_df.index == x]['FWER p-val'].values[0] == 0:
            pval = 0.001
            pval_annotation = '         p<'+str(pval)
        elif cluster6_mgstate_res_df[::-1][cluster6_mgstate_res_df.index == x]['FWER p-val'].values[0] < 0.01:
            pval = round(cluster6_mgstate_res_df[::-1][cluster6_mgstate_res_df.index == x]['FWER p-val'].values[0], 3)
            pval_annotation = '         p='+str(pval)
        else: 
            pval = round(cluster6_mgstate_res_df[::-1][cluster6_mgstate_res_df.index == x]['FWER p-val'].values[0], 2)
            pval_annotation = '         p='+str(pval)

        tick_spacing = 0.5
        ax2.yaxis.set_minor_locator(ticker.MultipleLocator(tick_spacing))
        ax2.tick_params(axis='y', which='major', right = False, labelright = False)
        ax2.tick_params(axis='y', which='minor', tick1On=False, tick2On=False)
        
        ax[x].set_yticks(ax[x].get_yticks())
        ax[x].set_yticklabels(labels = ['', '', name, '', ''])
        ax2.set_ylabel(pval_annotation, rotation = 0, va='center', size = 10)


In [ ]:
state_names = ['DAM', 'MIMS-iron', 'MIMS-foamy', 'LDAM', 'Mac_Polarization_A', 'Mac_Polarization_B', 'Mac_Polarization_C', 'Mac_Polarization_D']

pvals = []
for x in state_names:
    val = cluster6_mgstate_res_df[cluster6_mgstate_res_df.Term == x]['FWER p-val'].values[0]
    if val == 0:
        val = 0.001
    val = -1*np.log10(val)
    pvals.append(val)

data = [['DAM', 'MSi', 'MSf', 'LDAM', 'A', 'B', 'C', 'D'], 
        ('MG-C2', [
            pvals
        ])]

with rc_context({"figure.figsize": (4, 4)}): 
    N = len(data[0])
    theta = radar_factory(N, frame='polygon')
    
    spoke_labels = data.pop(0)
    title, case_data = data[0]
    
    fig, ax = plt.subplots(figsize=(4, 4), subplot_kw=dict(projection='radar'))
    fig.subplots_adjust(top=0.85, bottom=0.05)

    ax.set_rgrids([-1*np.log10(0.05), -1*np.log10(0.01), -1*np.log10(0.001)], labels = ["", "", ""], size = 10)
    ax.set_title(title,  position=(0.5, 1.1), ha='center', size = 20)

    ax.yaxis.grid(linestyle="dashed", alpha = 1, zorder = 1)
    ax.xaxis.grid(alpha = 0.1)
    ax.scatter(theta, case_data, s=20, c='tomato', zorder=10)
    
    for d in case_data:
        line = ax.plot(theta, d, color = "tomato", zorder = 5)
        ax.fill(theta, d,  alpha=0.75, color = "tomato", zorder = 4)
    ax.set_varlabels(spoke_labels)
    
    plt.show()

In [ ]:
# run GSEA Prerank test <-  Cluster 4
cluster4_mgstate_res = gseapy.prerank(rnk = cluster4_rank, 
        gene_sets=mg_genesets,
        threads = 48,
        permutation_num=1000,
        outdir=None,
        verbose = True,
        seed=0)

cluster4_mgstate_res_df = cluster4_mgstate_res.res2d

# examine significant GSEA results
cluster4_mgstate_res_df[(cluster4_mgstate_res_df['FWER p-val'] < 0.05)]

In [ ]:
pal = extract_colors_hexcodes(8, 'turbo')
categories = ['DAM', 'LDAM', 'MIMS-iron', 'MIMS-foamy', 'Mac_Polarization_A', 'Mac_Polarization_B', 'Mac_Polarization_C', 'Mac_Polarization_D']

categories_df = pd.DataFrame({'Term': categories, 'hexcode':pal[::-1]})

terms = cluster4_mgstate_res_df.iloc[::-1].Term
hits = [cluster4_mgstate_res.results[t]['hits'] for t in terms]
runes = [cluster4_mgstate_res.results[t]['RES'] for t in terms]

with rc_context({"figure.figsize": (3, 4), "grid.alpha":0}): 
    ax = gseaplot2(terms=terms, RESs=runes, hits=hits, 
                   rank_metric=cluster4_mgstate_res.ranking, 
                   colors = adjust_opacity(reorder_hexcodes(terms, categories_df), cluster4_mgstate_res_df),
                   figsize=(3,4))

    ax[8].set_title('MG-D', fontsize = 20, fontweight="bold")
    ax[0].set_xlabel('Gene Rank', fontsize = 13)
    ax[9].tick_params(axis="both", which="both", right=True, labelsize = 8)
    ax[8].set_ylabel('Enrichment Score', fontsize = 13)
    ax[9].set_ylabel('Rank Metric', fontsize = 13)
    ax[8].tick_params(axis="both", which="both", left=True, labelsize = 10)
    ax[8].get_legend().remove()

    terms.reset_index(drop=True, inplace=True)

    for x in terms.index.values:
        
        if terms[terms.index.values == x].values[0] == "Mac_Polarization_A":
            name = "IFN-I polarized"
        elif terms[terms.index.values == x].values[0] == "Mac_Polarization_B":
            name = "IFN-γ polarized"
        elif terms[terms.index.values == x].values[0] == "Mac_Polarization_C":
            name = "IL-1 polarized"
        elif terms[terms.index.values == x].values[0] == "Mac_Polarization_D":
            name = "TNF polarized"
        else:
            name = terms[terms.index.values == x].values[0]
        ax2 = ax[x].twinx()
        ax[x].tick_params(
            axis="both", which="both", bottom=False, top=False, right=True, left=True, labelleft = True, color = "white", labelsize = 10
        )

        if cluster4_mgstate_res_df[::-1][cluster4_mgstate_res_df.index == x]['FWER p-val'].values[0] == 0:
            pval = 0.001
            pval_annotation = '         p<'+str(pval)
        elif cluster4_mgstate_res_df[::-1][cluster4_mgstate_res_df.index == x]['FWER p-val'].values[0] < 0.01:
            pval = round(cluster4_mgstate_res_df[::-1][cluster4_mgstate_res_df.index == x]['FWER p-val'].values[0], 3)
            pval_annotation = '         p='+str(pval)
        else: 
            pval = round(cluster4_mgstate_res_df[::-1][cluster4_mgstate_res_df.index == x]['FWER p-val'].values[0], 2)
            pval_annotation = '         p='+str(pval)

        tick_spacing = 0.5
        ax2.yaxis.set_minor_locator(ticker.MultipleLocator(tick_spacing))
        ax2.tick_params(axis='y', which='major', right = False, labelright = False)
        ax2.tick_params(axis='y', which='minor', tick1On=False, tick2On=False)
        
        ax[x].set_yticks(ax[x].get_yticks())
        ax[x].set_yticklabels(labels = ['', '', name, '', ''])
        ax2.set_ylabel(pval_annotation, rotation = 0, va='center', size = 10)


In [ ]:
state_names = ['DAM', 'MIMS-iron', 'MIMS-foamy', 'LDAM', 'Mac_Polarization_A', 'Mac_Polarization_B', 'Mac_Polarization_C', 'Mac_Polarization_D']

pvals = []
for x in state_names:
    val = cluster4_mgstate_res_df[cluster4_mgstate_res_df.Term == x]['FWER p-val'].values[0]
    if val == 0:
        val = 0.001
    val = -1*np.log10(val)
    pvals.append(val)

data = [['DAM', 'MSi', 'MSf', 'LDAM', 'A', 'B', 'C', 'D'], 
        ('MG-D', [
            pvals
        ])]

with rc_context({"figure.figsize": (4, 4)}): 
    N = len(data[0])
    theta = radar_factory(N, frame='polygon')
    
    spoke_labels = data.pop(0)
    title, case_data = data[0]
    
    fig, ax = plt.subplots(figsize=(4, 4), subplot_kw=dict(projection='radar'))
    fig.subplots_adjust(top=0.85, bottom=0.05)

    ax.set_rgrids([-1*np.log10(0.05), -1*np.log10(0.01), -1*np.log10(0.001)], labels = ["", "", ""], size = 10)
    ax.set_title(title,  position=(0.5, 1.1), ha='center', size = 20)

    ax.yaxis.grid(linestyle="dashed", alpha = 1, zorder = 1)
    ax.xaxis.grid(alpha = 0.1)
    ax.scatter(theta, case_data, s=20, c='pink', zorder=10)
    
    for d in case_data:
        line = ax.plot(theta, d, color = "pink", zorder = 5)
        ax.fill(theta, d,  alpha=0.75, color = "pink", zorder = 4)
    ax.set_varlabels(spoke_labels)
    
    plt.show()

In [ ]:
# next, make heatmaps for microglia state GSEA results

In [ ]:
enriched_terms = [cluster2_mgstate_res_df[cluster2_mgstate_res_df['FWER p-val'] < 0.05].Term.tolist() +  
                  cluster5_mgstate_res_df[cluster5_mgstate_res_df['FWER p-val'] < 0.05].Term.tolist() + 
                  cluster1_mgstate_res_df[cluster1_mgstate_res_df['FWER p-val'] < 0.05].Term.tolist() + 
                  cluster6_mgstate_res_df[cluster6_mgstate_res_df['FWER p-val'] < 0.05].Term.tolist() + 
                  cluster4_mgstate_res_df[cluster4_mgstate_res_df['FWER p-val'] < 0.05].Term.tolist()
                 ]

enriched_terms = enriched_terms[0]
enriched_terms = np.unique(enriched_terms)
enriched_terms

In [ ]:
enriched_terms = list(enriched_terms)

In [ ]:
mgstates = cluster2_mgstate_res_df.Term

In [ ]:
# aggreage GSEA results
mgstate_results_dict = {
    'Cluster2':cluster2_mgstate_res_df, 
    'Cluster5':cluster5_mgstate_res_df,
    'Cluster1':cluster1_mgstate_res_df,
    'Cluster6':cluster6_mgstate_res_df,
    'Cluster4':cluster4_mgstate_res_df,
               }
mgstate_df = []

for key in mgstate_results_dict:
    for y in list(range(0, len(mgstates))):
        mgstate_df.append(
        {
            'Comparison': key,
            'Set': mgstates[y],
            'NES': mgstate_results_dict[key][mgstate_results_dict[key].Term == mgstates[y]]['NES'].values[0],
            'pval': mgstate_results_dict[key][mgstate_results_dict[key].Term == mgstates[y]]['FWER p-val'].values[0]
        }
    )

mgstate_df = pd.DataFrame(mgstate_df)

In [ ]:
%%R -i mgstate_df
# load data frame in R
mgstate.df <- mgstate_df %>% mutate(pval = ifelse(pval == 0, 0.001, pval)) %>% mutate(score = NES*-log10(pval))

In [ ]:
%%R

scores.mat = reshape2::acast(mgstate.df, Set~Comparison, value.var = "score")

In [ ]:
%%R

sig_mat = reshape2::acast(mgstate.df, Set~Comparison, value.var = "pval")

In [ ]:
%%R -w 4 -h 4 -r 300 --units in
# plot gsea heatmap
hmap = Heatmap(scores.mat, name = "Enrichment", 
        col = circlize::colorRamp2(breaks = c(-5, -2.5, 0, 2.5, 5), 
                                   colors=rev(RColorBrewer::brewer.pal(5, "RdBu"))),
        row_labels = TeX(c('DAM',
                           'LDAM',
                           'IFN-I polarized',
                           'IFN-$\\gamma$ polarized',
                           'IL-1 polarized',
                           'TNF polarized',  
                            'MIMS-foamy',
                           'MIMS-iron')),
        cell_fun = function(j, i, x, y, w, h, fill){
            if(sig_mat[i, j] <= 0.001){
            	gb = textGrob("***")
            	gb_w = convertWidth(grobWidth(gb), "mm")
            	gb_h = convertHeight(grobHeight(gb), "mm")
            	grid.text("***", x, y - gb_h*0.9 + gb_w*0.4)
            } else if(sig_mat[i, j] < 0.01){
            	gb = textGrob("**")
            	gb_w = convertWidth(grobWidth(gb), "mm")
            	gb_h = convertHeight(grobHeight(gb), "mm")
            	grid.text("**", x, y - gb_h*0.7 + gb_w*0.4)
            } else if(sig_mat[i, j] < 0.05){
            	gb = textGrob("*")
            	gb_w = convertWidth(grobWidth(gb), "mm")
            	gb_h = convertHeight(grobHeight(gb), "mm")
            	grid.text("*", x, y - gb_h*0.5 + gb_w*0.4)
            } else {
                gb = textGrob("")
            	gb_w = convertWidth(grobWidth(gb), "mm")
            	gb_h = convertHeight(grobHeight(gb), "mm")
                grid.text("", x, y - gb_h*0.5 + gb_w*0.4)
            }
        },
        #column_title = "Astrocyte Cluster", 
        row_title = "Microglia State",
        row_title_gp = grid::gpar(fontsize = 12),
        row_names_gp = grid::gpar(fontsize = 10),
        heatmap_legend_param = list(direction = "horizontal",
                                   title_position = "topcenter"
                                   )
       )

draw(hmap)

In [ ]:
# save mgstate results
cluster1_mgstate_res_df.to_csv("../output/human/human_microglia_cluster1_v_cluster3_degs_mgstate_results.csv")
cluster1_mgstate_res_df.to_pickle("../output/human/human_microglia_cluster1_v_cluster3_degs_mgstate_results.pkl")

cluster2_mgstate_res_df.to_csv("../output/human/human_microglia_cluster2_v_cluster3_degs_mgstate_results.csv")
cluster2_mgstate_res_df.to_pickle("../output/human/human_microglia_cluster2_v_cluster3_degs_mgstate_results.pkl")

cluster5_mgstate_res_df.to_csv("../output/human/human_microglia_cluster5_v_cluster3_degs_mgstate_results.csv")
cluster5_mgstate_res_df.to_pickle("../output/human/human_microglia_cluster5_v_cluster3_degs_mgstate_results.pkl")

cluster6_mgstate_res_df.to_csv("../output/human/human_microglia_cluster6_v_cluster3_degs_mgstate_results.csv")
cluster6_mgstate_res_df.to_pickle("../output/human/human_microglia_cluster6_v_cluster3_degs_mgstate_results.pkl")

cluster4_mgstate_res_df.to_csv("../output/human/human_microglia_cluster4_v_cluster3_degs_mgstate_results.csv")
cluster4_mgstate_res_df.to_pickle("../output/human/human_microglia_cluster4_v_cluster3_degs_mgstate_results.pkl")

In [ ]:
%%R

# save aggregated results
saveRDS(mgstate.df, "../output/human/human_microglia_mgstates_gsea_results.rds")
write.csv(mgstate.df, "../output/human/human_microglia_mgstates_gsea_results.csv")